# Advanced Python — Senior Engineer / SDE-3

**Deep mechanics, modern idioms, and concurrency.**

This notebook assumes you already write Python fluently. It covers *how the language actually works* — the object model, the type system, the descriptor protocol, MRO, and the four concurrency models — rather than syntax.

**Modern-only.** Where Python has replaced an older approach, only the current one is shown:

| Old | Modern |
|---|---|
| `typing.List[int]`, `Optional[X]` | `list[int]`, `X \| None` |
| `TypeVar` + `Generic[T]` | `class Box[T]` (PEP 695, 3.12+) |
| `%` / `.format()` | f-strings |
| `os.path` | `pathlib` |
| metaclasses for simple hooks | `__init_subclass__`, `__set_name__` |
| `asyncio.gather` for fan-out | `asyncio.TaskGroup` (3.11+) |
| `lru_cache(maxsize=None)` | `functools.cache` |
| `setup.py` | `pyproject.toml` + `uv` |
| threads for I/O fan-out | `asyncio` |

## Contents

| # | Section |
|---|---|
| 1 | The data model (dunder protocols) |
| 2 | The modern type system |
| 3 | Dataclasses and value objects |
| 4 | OOP: attributes, MRO, multiple inheritance |
| 5 | Descriptors — the mechanism behind everything |
| 6 | ABCs vs Protocols; `__init_subclass__`; metaclasses |
| 7 | Iterators, generators, lazy pipelines |
| 8 | Context managers |
| 9 | Decorators and `functools` |
| 10 | Closures and scoping |
| 11 | Memory model, GC, weakrefs |
| 12 | **Concurrency: GIL, threading, processes, asyncio, free-threading** |
| 13 | Structural pattern matching |
| 14 | Exception groups and `except*` |
| 15 | Performance engineering |
| 16 | Tooling |
| 17 | Interview questions |


In [ ]:
import sys, platform

V = sys.version_info
print("Python  :", sys.version.split()[0])
print("Impl    :", platform.python_implementation())
print("Platform:", sys.platform)

# Feature gates used throughout this notebook
GATES = {
    "match/case (3.10)":            V >= (3, 10),
    "ExceptionGroup / except* (3.11)": V >= (3, 11),
    "asyncio.TaskGroup (3.11)":     V >= (3, 11),
    "Self type (3.11)":             V >= (3, 11),
    "PEP 695 generics (3.12)":      V >= (3, 12),
    "type statement (3.12)":        V >= (3, 12),
    "free-threaded build (3.13+)":  hasattr(sys, "_is_gil_enabled"),
    "concurrent.interpreters (3.14)": V >= (3, 14),
}
for name, ok in GATES.items():
    print(f"  {'OK ' if ok else '-- '} {name}")

# Free-threaded builds are a SEPARATE binary (python3.14t), not a runtime flag.
if hasattr(sys, "_is_gil_enabled"):
    print("\nGIL enabled:", sys._is_gil_enabled())
else:
    print("\nGIL: always on in this build")

---

# 1. The data model (dunder protocols)

Python has no interfaces enforced by the compiler. Instead, **behaviour is protocol-based**: `len(x)` calls `x.__len__()`, `x + y` calls `x.__add__(y)`, `for i in x` calls `x.__iter__()`. Implement the dunder, get the syntax.

Understanding this is what separates "uses Python" from "designs in Python".

## 1.1 Object protocol: `__repr__`, `__str__`, `__format__`

In [ ]:
from decimal import Decimal

class Money:
    __slots__ = ("amount", "currency")          # see section 5.4

    def __init__(self, amount, currency="INR"):
        self.amount = Decimal(str(amount))
        self.currency = currency

    def __repr__(self):
        # For DEVELOPERS. Should be unambiguous and, ideally, eval-able.
        return f"Money({self.amount!r}, {self.currency!r})"

    def __str__(self):
        # For USERS. Falls back to __repr__ if not defined.
        return f"{self.currency} {self.amount:,.2f}"

    def __format__(self, spec):
        # Powers f"{obj:spec}" — this is why format specs work on datetimes
        if spec == "short":
            return f"{self.currency}{self.amount:.0f}"
        return format(str(self), spec)

m = Money("1234.5")
print(repr(m))
print(str(m))
print(f"{m}")
print(f"{m:short}")
print(f"{m:>20}")
print([m, m])           # containers use __repr__ on their elements

> **`__repr__` is not optional in production code.** Logs, tracebacks, debugger watches and `list` displays all use it. A default `<Money object at 0x7f...>` in a log line costs real debugging time.

## 1.2 The `__eq__` / `__hash__` contract

**The rule:** if `a == b` then `hash(a) == hash(b)`. Break it and dicts/sets silently misbehave.

Defining `__eq__` sets `__hash__ = None`, making instances unhashable — that's Python protecting you.

In [ ]:
class BadPoint:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return isinstance(other, BadPoint) and (self.x, self.y) == (other.x, other.y)
    # no __hash__ defined

p = BadPoint(1, 2)
print(BadPoint(1, 2) == BadPoint(1, 2))     # True
try:
    {p}
except TypeError as e:
    print("TypeError:", e)                   # unhashable — __hash__ was set to None

In [ ]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented          # let Python try the reflected operation
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))      # hash the SAME fields you compare

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

a, b = Point(1, 2), Point(1, 2)
print(a == b, hash(a) == hash(b))
print({a, b})                              # one element — they're equal AND hash equal
print({a: "first", b: "second"})           # b overwrites a's value

> **Returning `NotImplemented` (not `NotImplementedError`) from `__eq__`** lets Python fall back to the other operand's `__eq__`, and finally to identity. Returning `False` instead breaks symmetric comparison with subclasses and unrelated types.

**Mutability rule:** anything used as a dict key or set member must be **immutably hashable**. If you hash mutable fields and then mutate, the object becomes unfindable in its own dict.

In [ ]:
class MutableKey:
    def __init__(self, v): self.v = v
    def __eq__(self, o): return isinstance(o, MutableKey) and self.v == o.v
    def __hash__(self): return hash(self.v)

k = MutableKey(1)
d = {k: "value"}
print(d.get(k))          # 'value'

k.v = 2                  # mutate a hashed field
print(d.get(k))          # None  <<< the object is LOST inside its own dict
print(list(d.items()))   # it's still there, just in the wrong bucket

## 1.3 Ordering: `__lt__` and `total_ordering`

Define `__lt__` + `__eq__`, then let `functools.total_ordering` synthesise the rest.

In [ ]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, s):
        self.parts = tuple(int(p) for p in s.split("."))
    def __eq__(self, o):
        return isinstance(o, Version) and self.parts == o.parts
    def __lt__(self, o):
        if not isinstance(o, Version): return NotImplemented
        return self.parts < o.parts
    def __hash__(self): return hash(self.parts)
    def __repr__(self): return "v" + ".".join(map(str, self.parts))

vs = [Version("1.10.0"), Version("1.2.0"), Version("2.0.0"), Version("1.2.10")]
print(sorted(vs))
print(max(vs), Version("1.2.0") <= Version("1.2.10"))   # <= synthesised

## 1.4 Container protocol

`__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__`.

**Key fact:** `__getitem__` alone gives you iteration and `in` for free (the legacy sequence protocol), but implementing `__iter__` and `__contains__` explicitly is faster and clearer.

In [ ]:
class OrderLines:
    def __init__(self):
        self._lines = {}

    def __len__(self):                      # len(obj)
        return len(self._lines)

    def __getitem__(self, key):             # obj[key]  — also enables slicing if you handle it
        return self._lines[key]

    def __setitem__(self, key, value):      # obj[key] = value
        self._lines[key] = value

    def __delitem__(self, key):             # del obj[key]
        del self._lines[key]

    def __contains__(self, key):            # key in obj  — O(1) instead of scanning
        return key in self._lines

    def __iter__(self):                     # for k in obj
        return iter(self._lines)

    def __bool__(self):                     # truthiness. Without it, Python falls back to __len__
        return bool(self._lines)

    def __repr__(self):
        return f"OrderLines({self._lines!r})"

ol = OrderLines()
ol["S18_1749"] = 30
ol["S18_2248"] = 50
print(len(ol), "S18_1749" in ol, bool(ol))
print(list(ol), ol["S18_2248"])
del ol["S18_2248"]
print(ol)

## 1.5 Numeric protocol and reflected operations

For `a + b`, Python tries `a.__add__(b)`. If that returns `NotImplemented`, it tries `b.__radd__(a)`. In-place `a += b` tries `a.__iadd__(b)` first.

In [ ]:
class Vec:
    __slots__ = ("x", "y")
    def __init__(self, x, y): self.x, self.y = x, y
    def __repr__(self): return f"Vec({self.x}, {self.y})"

    def __add__(self, o):
        if isinstance(o, Vec):  return Vec(self.x + o.x, self.y + o.y)
        return NotImplemented

    def __mul__(self, k):
        if isinstance(k, (int, float)): return Vec(self.x * k, self.y * k)
        return NotImplemented

    def __rmul__(self, k):           # enables  3 * vec  (int has no __mul__ for Vec)
        return self.__mul__(k)

    def __neg__(self):  return Vec(-self.x, -self.y)
    def __abs__(self):  return (self.x ** 2 + self.y ** 2) ** 0.5
    def __eq__(self, o): return isinstance(o, Vec) and (self.x, self.y) == (o.x, o.y)

v = Vec(3, 4)
print(v + Vec(1, 1))
print(v * 2)
print(3 * v)              # __rmul__ path
print(-v, abs(v))
print(v + 1 if False else "Vec + int returns NotImplemented -> TypeError")

## 1.6 Callable, and controlling attribute access

| Dunder | Fires when |
|---|---|
| `__call__` | `obj(...)` |
| `__getattr__` | attribute **not found** normally — a fallback |
| `__getattribute__` | **every** attribute access (dangerous, rarely correct) |
| `__setattr__` | every assignment |
| `__dir__` | `dir(obj)` |

In [ ]:
class Config:
    """Attribute-style access over a nested dict, with a call interface."""

    def __init__(self, data):
        # Bypass our own __setattr__ to avoid infinite recursion
        object.__setattr__(self, "_data", data)

    def __getattr__(self, name):
        # Only called when normal lookup FAILS — so _data is found normally
        try:
            value = self._data[name]
        except KeyError:
            raise AttributeError(f"no config key {name!r}") from None
        return Config(value) if isinstance(value, dict) else value

    def __setattr__(self, name, value):
        raise AttributeError("Config is read-only")

    def __call__(self, key, default=None):
        return self._data.get(key, default)

    def __dir__(self):
        return sorted(self._data)

cfg = Config({"db": {"host": "localhost", "port": 3306}, "debug": True})
print(cfg.db.host, cfg.db.port, cfg.debug)
print(cfg("missing", "fallback"))
print(dir(cfg))
for bad in ("cfg.nope", "cfg.debug = False"):
    try: eval(bad) if "=" not in bad else exec(bad)
    except AttributeError as e: print(f"{bad} -> AttributeError: {e}")

> **`__getattr__` vs `__getattribute__`:** `__getattr__` is a *fallback* and cheap. `__getattribute__` intercepts **everything**, including `self._data`, so any naive implementation recurses infinitely. Reach for `__getattr__` in ~99% of cases.

## 1.7 The full protocol map

| Protocol | Dunders |
|---|---|
| Representation | `__repr__` `__str__` `__format__` `__bytes__` |
| Equality / hashing | `__eq__` `__hash__` |
| Ordering | `__lt__` `__le__` `__gt__` `__ge__` |
| Truthiness | `__bool__` (falls back to `__len__`) |
| Container | `__len__` `__getitem__` `__setitem__` `__delitem__` `__contains__` |
| Iteration | `__iter__` `__next__` `__reversed__` |
| Numeric | `__add__` `__sub__` `__mul__` `__truediv__` `__mod__` `__pow__` + `__r*__` + `__i*__` |
| Unary | `__neg__` `__pos__` `__abs__` `__round__` `__index__` |
| Callable | `__call__` |
| Attributes | `__getattr__` `__getattribute__` `__setattr__` `__delattr__` `__dir__` |
| Descriptors | `__get__` `__set__` `__delete__` `__set_name__` |
| Context | `__enter__` `__exit__` `__aenter__` `__aexit__` |
| Async | `__await__` `__aiter__` `__anext__` |
| Copy / pickle | `__copy__` `__deepcopy__` `__reduce__` `__getstate__` `__setstate__` |
| Class creation | `__init_subclass__` `__class_getitem__` `__mro_entries__` |


---

# 2. The modern type system

Type hints are **not enforced at runtime**. They exist for static checkers (mypy, pyright), IDEs, and documentation. Their value at senior level is in *API design* — they encode contracts.

## 2.1 Modern syntax only

In [ ]:
from collections.abc import Callable, Iterable, Sequence, Mapping, Iterator
from typing import Any, Final, Literal, TypedDict, NewType, overload

# 3.10+ : builtin generics and | unions. Never import List/Dict/Optional.
def parse(rows: list[str], sep: str = ",") -> dict[str, list[str]]:
    out: dict[str, list[str]] = {}
    for r in rows:
        k, *rest = r.split(sep)
        out[k] = rest
    return out

def find(items: Sequence[int], target: int) -> int | None:      # NOT Optional[int]
    for i, v in enumerate(items):
        if v == target:
            return i
    return None

print(parse(["a,1,2", "b,3,4"]))
print(find([1, 2, 3], 2), find([1, 2, 3], 9))

> **Argument types should be as general as possible; return types as specific as possible.**
> Accept `Iterable[str]` / `Sequence[str]` / `Mapping[str, int]`; return `list[str]` / `dict[str, int]`.
> Import the abstract ones from `collections.abc`, not `typing`.

In [ ]:
from collections.abc import Iterable, Sequence, Mapping

# BAD: over-constrains the caller
def total_bad(values: list[float]) -> float:
    return sum(values)

# GOOD: accepts lists, tuples, generators, sets, dict.values()...
def total(values: Iterable[float]) -> float:
    return sum(values)

print(total([1.0, 2.0]), total((1.0, 2.0)), total(x * 1.0 for x in range(4)))

# Read-only mapping input, concrete dict output
def counts(m: Mapping[str, int]) -> dict[str, int]:
    return {k: v * 2 for k, v in m.items()}
print(counts({"a": 1}))

## 2.2 PEP 695 generics (3.12+)

The `TypeVar` + `Generic[T]` boilerplate is gone. Type parameters are declared inline.

In [ ]:
import sys
if sys.version_info >= (3, 12):
    ns = {}
    exec('''
from collections.abc import Callable, Iterable

# Generic function
def first[T](items: Iterable[T]) -> T | None:
    for x in items:
        return x
    return None

# Generic class
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()
    def __len__(self) -> int:
        return len(self._items)

# Bounded and constrained type params
def largest[T: (int, float)](items: Iterable[T]) -> T:
    return max(items)

# Type alias statement (3.12+) — lazily evaluated, no forward-ref quoting needed
type Json = None | bool | int | float | str | list[Json] | dict[str, Json]
type Handler[T] = Callable[[T], None]

# Generic over a ParamSpec + return type, no imports needed
def logged[**P, R](fn: Callable[P, R]) -> Callable[P, R]:
    def inner(*args: P.args, **kwargs: P.kwargs) -> R:
        return fn(*args, **kwargs)
    return inner
''', ns)
    print("first    :", ns["first"]([10, 20]), ns["first"]([]))
    s = ns["Stack"]()
    s.push("a"); s.push("b")
    print("Stack    :", len(s), s.pop())
    print("largest  :", ns["largest"]([3, 9, 4]))
    print("Json alias:", ns["Json"])
else:
    print("Requires 3.12+. Pre-3.12 equivalent:")
    print("  T = TypeVar('T');  class Stack(Generic[T]): ...")

## 2.3 Protocols — structural typing

`Protocol` is duck typing the static checker can verify. **This is usually the right abstraction, not ABC** — see section 6.1.

In [ ]:
from typing import Protocol, runtime_checkable

class Writer(Protocol):
    def write(self, data: str) -> int: ...

@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...

def emit(sink: Writer, rows: list[str]) -> int:
    """Accepts ANY object with a matching write() — no inheritance required."""
    return sum(sink.write(r + "\n") for r in rows)

import io
buf = io.StringIO()
print("bytes written:", emit(buf, ["a", "b"]))
print(repr(buf.getvalue()))

class Collector:                      # no base class, no registration
    def __init__(self): self.rows = []
    def write(self, data: str) -> int:
        self.rows.append(data); return len(data)
    def close(self) -> None: ...

c = Collector()
emit(c, ["x", "y"])
print(c.rows)
print("isinstance via runtime_checkable:", isinstance(c, Closeable))

> `@runtime_checkable` makes `isinstance()` work, but it only checks **method names, not signatures**. Treat it as a smoke test, not a guarantee.

## 2.4 Precision tools: `Literal`, `TypedDict`, `NewType`, `overload`, `Final`, `Annotated`

In [ ]:
from typing import Literal, TypedDict, NewType, Final, Annotated, overload, Self, NotRequired

# Literal: restrict to exact values (checkers catch typos)
Mode = Literal["append", "overwrite", "error"]
def save(path: str, mode: Mode = "error") -> None: ...

# TypedDict: shape of a dict, keys known statically
class OrderLine(TypedDict):
    orderNumber: int
    productCode: str
    qty: int
    comment: NotRequired[str]          # optional key

line: OrderLine = {"orderNumber": 10100, "productCode": "S18_1749", "qty": 30}
print(line)

# NewType: a distinct type at check time, zero runtime cost
CustomerId = NewType("CustomerId", int)
OrderId    = NewType("OrderId", int)
def lookup(cid: CustomerId) -> None: ...
lookup(CustomerId(103))                # a plain int would be flagged by a checker

# Final: cannot be reassigned
MAX_RETRIES: Final = 3

# Annotated: attach metadata (Pydantic, FastAPI, msgspec all read this)
Port = Annotated[int, "range 1-65535"]

print(f"Mode={Mode}, MAX_RETRIES={MAX_RETRIES}, Port={Port}")

In [ ]:
from typing import overload

# @overload declares multiple signatures for one implementation.
# Only the final def has a body; the overloads are erased at runtime.
@overload
def get(key: str) -> str | None: ...
@overload
def get(key: str, default: str) -> str: ...
def get(key, default=None):
    return {"a": "1"}.get(key, default)

print(get("a"), get("zz"), get("zz", "fallback"))
# A checker knows get("zz") is `str | None` but get("zz", "x") is `str`.

In [ ]:
from typing import Self

# Self (3.11+) types fluent APIs and alternative constructors correctly under subclassing
class Query:
    def __init__(self) -> None:
        self._where: list[str] = []

    def where(self, clause: str) -> Self:      # NOT -> "Query"
        self._where.append(clause)
        return self

    @classmethod
    def create(cls) -> Self:
        return cls()

    def build(self) -> str:
        return "SELECT * FROM t" + (" WHERE " + " AND ".join(self._where) if self._where else "")

class MySQLQuery(Query):
    def limit(self, n: int) -> Self:
        self._where.append(f"-- LIMIT {n}")
        return self

# Without Self, the checker would think .where() returns Query and lose .limit()
print(MySQLQuery.create().where("a = 1").limit(10).build())

## 2.5 Runtime validation is a separate concern

Type hints do nothing at runtime. For validating external input (API payloads, config, CSV rows) use a validation library that *reads* the hints:

| Tool | Use for |
|---|---|
| **Pydantic v2** | API/config models, coercion, JSON schema. Rust core, fast. |
| **msgspec** | Fastest serialise/validate; great for high-throughput services |
| **attrs** | Rich class building with validators/converters |
| stdlib `dataclasses` | Plain value objects, no validation |

Design rule: **validate at the boundary, then trust your types inside**.

---

# 3. Dataclasses and value objects

## 3.1 Modern dataclass options

In [ ]:
from dataclasses import dataclass, field, replace, asdict, astuple, fields, KW_ONLY

@dataclass(slots=True, frozen=True, kw_only=True, order=True)
class OrderLine:
    order_number: int
    product_code: str
    qty: int = 1
    price: float = 0.0
    tags: tuple[str, ...] = ()                       # immutable default is fine
    meta: dict[str, str] = field(default_factory=dict, compare=False, repr=False)

    def __post_init__(self) -> None:
        # frozen=True blocks normal assignment, so validate and use object.__setattr__
        if self.qty <= 0:
            raise ValueError("qty must be positive")

    @property
    def total(self) -> float:
        return self.qty * self.price

a = OrderLine(order_number=10100, product_code="S18_1749", qty=30, price=136.0)
print(a)
print("total:", a.total)
print("hashable:", hash(a) is not None)

b = replace(a, qty=40)              # the frozen-safe way to "modify"
print(b.qty, a.qty)

try:
    a.qty = 99
except Exception as e:
    print(type(e).__name__, ":", e)

try:
    OrderLine(order_number=1, product_code="x", qty=0)
except ValueError as e:
    print("ValueError:", e)

### What each flag buys you

| Flag | Effect | When |
|---|---|---|
| `slots=True` | no `__dict__`; ~30-50% less memory, faster attribute access | **default choice** for value objects |
| `frozen=True` | immutable + auto `__hash__` | anything used as a dict key, or shared across threads |
| `kw_only=True` | callers must name arguments | 3+ fields, or fields that could be transposed |
| `order=True` | generates `__lt__` etc. from field order | sortable records |
| `eq=False` | keeps identity semantics | entities with an ID, not values |

`field(...)` options: `default_factory` (mutable defaults), `compare=False` (exclude from `==`), `repr=False` (hide from logs — use for secrets), `init=False` (computed in `__post_init__`), `metadata` (arbitrary dict).

In [ ]:
from dataclasses import dataclass, field, fields, asdict
import sys

@dataclass
class WithDict:
    a: int; b: int; c: int

@dataclass(slots=True)
class WithSlots:
    a: int; b: int; c: int

w1, w2 = WithDict(1,2,3), WithSlots(1,2,3)
print("dict-based  :", sys.getsizeof(w1) + sys.getsizeof(w1.__dict__), "bytes")
print("slots-based :", sys.getsizeof(w2), "bytes (no __dict__)")
print("has __dict__:", hasattr(w1, "__dict__"), hasattr(w2, "__dict__"))

# Introspection — this is how serialisers and ORMs work
print([f.name for f in fields(WithSlots)])
print(asdict(w1))

> **`slots=True` caveats:** you cannot add attributes not declared as fields, and multiple inheritance from two slotted classes with overlapping names fails. Both are usually what you want.

## 3.2 Choosing a value-object type

| | Mutable | Hashable | Memory | Validation | Use when |
|---|---|---|---|---|---|
| `@dataclass(slots=True)` | yes | no | low | manual | internal mutable record |
| `@dataclass(frozen=True, slots=True)` | no | yes | low | manual | **value object, default choice** |
| `NamedTuple` | no | yes | lowest | none | tuple-compatible, index access needed |
| `TypedDict` | yes | n/a | dict | none | JSON-shaped data, no methods |
| Pydantic `BaseModel` | configurable | configurable | higher | **automatic** | external input boundary |

Rule of thumb: **frozen slotted dataclass inside the system, Pydantic at the edges.**

In [ ]:
from typing import NamedTuple

class Coord(NamedTuple):
    lat: float
    lon: float
    label: str = ""

c = Coord(22.8, 86.2, "Ranchi")
print(c, c.lat, c[0])                  # attribute AND index access
print(tuple(c), c._replace(label="X")) # tuple-compatible; _replace like dataclasses.replace
lat, lon, label = c                    # unpacks
print(lat, lon, label)

---

# 4. OOP: attributes, MRO, multiple inheritance

## 4.1 Attribute lookup order

When you access `obj.x`, Python walks a precise sequence. Knowing it explains every "why isn't my property firing" bug.

```
1. type(obj).__mro__  -> is there a DATA descriptor (has __set__ or __delete__)?   -> use it
2. obj.__dict__       -> instance attribute                                        -> use it
3. type(obj).__mro__  -> class attribute / NON-data descriptor (only __get__)       -> use it
4. type(obj).__getattr__  -> fallback hook
5. AttributeError
```

**Data descriptors beat the instance dict. The instance dict beats non-data descriptors.** That single rule is why `@property` cannot be shadowed by an instance attribute, but a method can.

In [ ]:
class Demo:
    class_attr = "from class"

    @property
    def prop(self):                      # property = DATA descriptor (has __set__)
        return "from property"

    def method(self):                    # function = NON-data descriptor (only __get__)
        return "from method"

d = Demo()

# Instance dict wins over a plain class attribute
d.class_attr = "from instance"
print(d.class_attr, "|", Demo.class_attr)

# Instance dict wins over a method (step 2 beats step 3)
d.__dict__["method"] = lambda: "shadowed!"
print(d.method())

# But a property CANNOT be shadowed — step 1 beats step 2
d.__dict__["prop"] = "shadow attempt"
print(d.prop)                            # still 'from property'
print("in __dict__ but ignored:", d.__dict__["prop"])

## 4.2 Class attributes vs instance attributes — the mutable class attribute bug

In [1]:
class Registry:
    items = []                    # <<< SHARED by every instance

    def add(self, x):
        self.items.append(x)      # mutates the CLASS list

a, b = Registry(), Registry()
a.add(1); b.add(2)
print("a:", a.items, "| b:", b.items, "| class:", Registry.items)
print("same object:", a.items is b.items)

a: [1, 2] | b: [1, 2] | class: [1, 2]
same object: True


In [ ]:
class Registry:
    def __init__(self):
        self.items: list = []     # per-instance

a, b = Registry(), Registry()
a.items.append(1); b.items.append(2)
print("a:", a.items, "| b:", b.items, "| same object:", a.items is b.items)

> Same root cause as the mutable default argument: **class bodies execute once**, at class-creation time.

## 4.3 MRO and C3 linearisation

With multiple inheritance, Python computes a single **method resolution order** using the C3 algorithm. The guarantees:

1. A class always precedes its parents
2. Parents appear in the order written in the class statement
3. The order is **monotonic** — consistent across the whole hierarchy

If no such order exists, the class statement raises `TypeError` at definition time.

In [ ]:
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B -> " + super().who()
class C(A):
    def who(self): return "C -> " + super().who()
class D(B, C):
    def who(self): return "D -> " + super().who()

print([c.__name__ for c in D.__mro__])
print(D().who())

### The diamond, and why `super()` is not "the parent"

`D.__mro__` is `[D, B, C, A, object]`. Inside `B.who`, `super()` resolves to **C**, not A — because `super()` means *"the next class in the MRO of `type(self)`"*, not *"my base class"*.

This is the single most misunderstood thing about Python OOP. `A.who` runs **once**, not twice, which is exactly what you want for cooperative `__init__` chains.

In [ ]:
# Proof that super() depends on the INSTANCE type, not the defining class
print("B alone :", B().who())        # B -> A     (MRO of B is [B, A, object])
print("D       :", D().who())        # D -> B -> C -> A
print("MRO(B)  :", [c.__name__ for c in B.__mro__])

In [ ]:
# An inconsistent hierarchy is rejected at class-creation time
class X: pass
class Y: pass
class P(X, Y): pass
class Q(Y, X): pass
try:
    class Bad(P, Q): pass            # P wants X before Y; Q wants Y before X
except TypeError as e:
    print("TypeError:", e)

## 4.4 Cooperative multiple inheritance — doing it correctly

The contract every class in a cooperative hierarchy must honour:

1. Call `super().__init__(**kwargs)` — **always**, even if you think you're last
2. Accept and forward `**kwargs`
3. Consume only your own arguments
4. `object.__init__` sits at the end and accepts nothing extra, so the chain must be clean by then

In [ ]:
class Base:
    def __init__(self, **kwargs):
        # Absorbs the tail of the chain; object.__init__ takes no args
        super().__init__()
        self.trace = ["Base"]

class Timestamped(Base):
    def __init__(self, *, created_at=None, **kwargs):
        super().__init__(**kwargs)                 # forward the rest
        self.created_at = created_at or "2026-01-01"
        self.trace.append("Timestamped")

class Auditable(Base):
    def __init__(self, *, author="system", **kwargs):
        super().__init__(**kwargs)
        self.author = author
        self.trace.append("Auditable")

class Versioned(Base):
    def __init__(self, *, version=1, **kwargs):
        super().__init__(**kwargs)
        self.version = version
        self.trace.append("Versioned")

class Document(Timestamped, Auditable, Versioned):
    def __init__(self, title, **kwargs):
        super().__init__(**kwargs)
        self.title = title
        self.trace.append("Document")

doc = Document("Q4 Report", author="aditi", version=3, created_at="2026-05-01")
print([c.__name__ for c in Document.__mro__])
print("trace  :", doc.trace)              # reverse MRO order — the chain unwinds
print("fields :", doc.title, doc.author, doc.version, doc.created_at)

> **Why the trace is in reverse MRO order:** each `__init__` calls `super().__init__()` *first*, so the deepest class finishes first and the outermost finishes last. If you need setup to happen before delegation, do it before the `super()` call — but be deliberate about it.

**Keyword-only arguments (`*,`) are essential here.** Positional arguments cannot be safely forwarded through an unknown-length MRO.

## 4.5 Mixins — the practical use of multiple inheritance

A mixin adds behaviour, holds no state of its own (or minimal state), and is never instantiated alone. This is 95% of legitimate multiple inheritance.

In [ ]:
import json
from datetime import datetime, timezone

class ReprMixin:
    """Generates __repr__ from __slots__ or __dict__."""
    def __repr__(self):
        names = getattr(self, "__slots__", None) or vars(self).keys()
        body = ", ".join(f"{n}={getattr(self, n)!r}" for n in names)
        return f"{type(self).__name__}({body})"

class JsonMixin:
    def to_json(self, **kw) -> str:
        names = getattr(self, "__slots__", None) or vars(self).keys()
        return json.dumps({n: getattr(self, n) for n in names}, default=str, **kw)

class ComparableByKeyMixin:
    """Requires the concrete class to define _sort_key()."""
    def _sort_key(self):
        raise NotImplementedError
    def __lt__(self, other): return self._sort_key() < other._sort_key()
    def __eq__(self, other): return self._sort_key() == other._sort_key()
    def __hash__(self):      return hash(self._sort_key())


class Order(ReprMixin, JsonMixin, ComparableByKeyMixin):
    __slots__ = ("number", "amount", "placed_at")

    def __init__(self, number, amount):
        self.number, self.amount = number, amount
        self.placed_at = datetime(2003, 1, 6, tzinfo=timezone.utc)

    def _sort_key(self):
        return (self.amount, self.number)

orders = [Order(10100, 500.0), Order(10101, 1500.0), Order(10102, 900.0)]
print(orders[0])
print(orders[0].to_json())
print(sorted(orders))

### Mixin ordering rule

**Mixins go left, the concrete base goes right:** `class Order(MixinA, MixinB, Base)`.

Because MRO is left-to-right, anything on the left can override and can call `super()` into what's on the right. Put the base first and your mixins become unreachable.

## 4.6 Composition over inheritance

Inheritance couples you to a parent's *implementation*. Prefer composition unless the subtype relationship is genuine (Liskov: any code working with the base must work with the subclass).

In [ ]:
from typing import Protocol

# --- Inheritance-based: rigid. Adding a channel means a new subclass;
#     combining two means multiple inheritance.
class NotifierBase:
    def send(self, msg): raise NotImplementedError

class EmailNotifier(NotifierBase):
    def send(self, msg): return f"email: {msg}"


# --- Composition-based: channels combine freely at runtime.
class Channel(Protocol):
    def deliver(self, msg: str) -> str: ...

class Email:
    def deliver(self, msg): return f"email: {msg}"
class Slack:
    def deliver(self, msg): return f"slack: {msg}"
class SMS:
    def deliver(self, msg): return f"sms:   {msg}"

class Notifier:
    def __init__(self, *channels: Channel):
        self.channels = channels
    def send(self, msg: str) -> list[str]:
        return [c.deliver(msg) for c in self.channels]

n = Notifier(Email(), Slack(), SMS())
for line in n.send("Deploy finished"):
    print(line)

# Swap behaviour at runtime — impossible with inheritance
n.channels = (Slack(),)
print(n.send("Quiet mode"))

**Decision heuristic**

| Use inheritance when | Use composition when |
|---|---|
| A genuine "is-a" that satisfies Liskov | "has-a" / "uses-a" |
| Sharing a stable, small interface | Behaviour varies independently |
| Framework requires it (`Exception`, `Enum`) | You want runtime swapping |
| Adding cross-cutting behaviour via mixins | Combinations would explode subclass count |


---

# 5. Descriptors — the mechanism behind everything

A descriptor is any object defining `__get__`, `__set__` or `__delete__`, used as a **class attribute**. Methods, `property`, `classmethod`, `staticmethod`, `functools.cached_property`, ORM columns and dataclass fields are all descriptors.

| Kind | Defines | Priority |
|---|---|---|
| **Data descriptor** | `__set__` and/or `__delete__` | beats the instance dict |
| **Non-data descriptor** | only `__get__` | loses to the instance dict |

## 5.1 `property` is just a data descriptor

In [ ]:
class Temperature:
    def __init__(self, celsius=0.0):
        self.celsius = celsius            # goes through the setter

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self) -> float:        # computed, read-only
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(t.celsius, t.fahrenheit)
t.celsius = 30
print(t.celsius, t.fahrenheit)

for bad in (-300,):
    try: t.celsius = bad
    except ValueError as e: print("ValueError:", e)

try: t.fahrenheit = 100
except AttributeError as e: print("AttributeError:", e)

print("property is a descriptor:", hasattr(Temperature.celsius, "__set__"))

> **Don't add a property until you need it.** Start with a plain attribute; converting to a property later is a non-breaking change for callers. That's the whole reason Python has no getters/setters convention.

## 5.2 Writing a reusable descriptor with `__set_name__`

`__set_name__(self, owner, name)` fires at class creation and tells the descriptor which attribute name it was bound to — no more passing the name manually.

In [ ]:
class Validated:
    """Reusable typed+validated attribute. One class, any number of fields."""

    def __init__(self, *, type_=None, min_value=None, max_value=None, choices=None):
        self.type_, self.min_value, self.max_value, self.choices = type_, min_value, max_value, choices

    def __set_name__(self, owner, name):
        self.name = name
        self.private = f"_{name}"          # where the real value lives

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self                     # accessed on the CLASS, not an instance
        return getattr(obj, self.private)

    def __set__(self, obj, value):
        if self.type_ and not isinstance(value, self.type_):
            raise TypeError(f"{self.name} must be {self.type_.__name__}, got {type(value).__name__}")
        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"{self.name} must be >= {self.min_value}")
        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"{self.name} must be <= {self.max_value}")
        if self.choices and value not in self.choices:
            raise ValueError(f"{self.name} must be one of {self.choices}")
        setattr(obj, self.private, value)


class Product:
    code   = Validated(type_=str)
    price  = Validated(type_=float, min_value=0.0)
    qty    = Validated(type_=int, min_value=0, max_value=10_000)
    line   = Validated(choices={"Classic Cars", "Motorcycles", "Planes"})

    def __init__(self, code, price, qty, line):
        self.code, self.price, self.qty, self.line = code, price, qty, line

    def __repr__(self):
        return f"Product({self.code!r}, {self.price}, {self.qty}, {self.line!r})"

p = Product("S18_1749", 136.0, 30, "Classic Cars")
print(p)

for attr, value in [("price", -1.0), ("qty", 99_999), ("line", "Boats"), ("price", "cheap")]:
    try:
        setattr(p, attr, value)
    except (ValueError, TypeError) as e:
        print(f"{attr}={value!r:<15} -> {type(e).__name__}: {e}")

> **Why `if obj is None: return self`** — when accessed on the class (`Product.price`), there's no instance, so the descriptor returns itself. That's how `help()`, `inspect` and IDEs introspect properties.

## 5.3 `cached_property` and lazy computation

In [ ]:
from functools import cached_property
import time

class Dataset:
    def __init__(self, n): self.n = n

    @cached_property
    def expensive_stat(self):
        """Computed once, then cached in the INSTANCE dict."""
        time.sleep(0.2)
        return sum(range(self.n))

d = Dataset(1_000_00)
t0 = time.perf_counter(); d.expensive_stat; first = time.perf_counter() - t0
t0 = time.perf_counter(); d.expensive_stat; second = time.perf_counter() - t0
print(f"first : {first:.3f}s")
print(f"second: {second:.8f}s   (read straight from __dict__)")
print("cached in instance dict:", "expensive_stat" in d.__dict__)

del d.__dict__["expensive_stat"]           # invalidate
print("after invalidation:", "expensive_stat" in d.__dict__)

> **`cached_property` is a NON-data descriptor** — it has `__get__` but no `__set__`. That's precisely how the caching works: the first access writes to `obj.__dict__`, and every later access finds the instance dict *first* (step 2 beats step 3) and never calls the descriptor again.
>
> **Two consequences:** it does not work with `__slots__` (no instance dict), and it is **not thread-safe** in current versions — two threads can both compute it. Guard with a lock if that matters.

## 5.4 `__slots__` in depth

In [ ]:
import sys

class WithDict:
    def __init__(self, a, b): self.a, self.b = a, b

class WithSlots:
    __slots__ = ("a", "b")
    def __init__(self, a, b): self.a, self.b = a, b

n = 100_000
d_objs = [WithDict(i, i) for i in range(n)]
s_objs = [WithSlots(i, i) for i in range(n)]

def deep_size(o):
    return sys.getsizeof(o) + (sys.getsizeof(o.__dict__) if hasattr(o, "__dict__") else 0)

print(f"WithDict  : {deep_size(d_objs[0]):>4} bytes/obj")
print(f"WithSlots : {deep_size(s_objs[0]):>4} bytes/obj")
print(f"saving over {n:,} objects: ~{(deep_size(d_objs[0]) - deep_size(s_objs[0])) * n / 1e6:.1f} MB")

# Slots block undeclared attributes — catches typos at runtime
s = WithSlots(1, 2)
try:
    s.c = 3
except AttributeError as e:
    print("AttributeError:", e)

**`__slots__` rules and gotchas**

- Subclasses need their **own** `__slots__`, or they regain a `__dict__` (and the saving)
- Declaring `__slots__ = ()` in a subclass keeps it slotted with no new fields
- Cannot multiply-inherit from two classes that both define non-empty `__slots__`
- Breaks `cached_property` and `functools.lru_cache` on instances
- Class-level defaults conflict: you cannot have `__slots__ = ("x",)` **and** `x = 0` in the same class
- Use it for: high-cardinality value objects, hot-path records, memory-constrained services. Skip it for: singletons, config objects, anything needing dynamic attributes

In [ ]:
class Parent:
    __slots__ = ("a",)

class ChildLeaky(Parent):        # no __slots__ -> gains __dict__
    pass

class ChildTight(Parent):
    __slots__ = ()               # stays slotted

print("leaky has __dict__:", hasattr(ChildLeaky(), "__dict__"))
print("tight has __dict__:", hasattr(ChildTight(), "__dict__"))

## 5.5 `classmethod` and `staticmethod` — what they're actually for

In [ ]:
from datetime import date

class Order:
    __slots__ = ("number", "lines", "placed")
    TAX_RATE = 0.18

    def __init__(self, number, lines, placed):
        self.number, self.lines, self.placed = number, lines, placed

    # ALTERNATIVE CONSTRUCTORS — the main reason classmethod exists.
    # `cls` means subclasses get instances of THEMSELVES, not of Order.
    @classmethod
    def from_csv_row(cls, row: str):
        num, *rest = row.split(",")
        return cls(int(num), [r for r in rest], date.today())

    @classmethod
    def empty(cls, number):
        return cls(number, [], date.today())

    # STATICMETHOD: logically belongs to the class, needs neither cls nor self.
    # It exists for namespacing and testability.
    @staticmethod
    def is_valid_code(code: str) -> bool:
        return len(code) > 4 and "_" in code

    def __repr__(self):
        return f"{type(self).__name__}(number={self.number}, lines={len(self.lines)})"

class PriorityOrder(Order):
    pass

print(Order.from_csv_row("10100,S18_1749,S18_2248"))
print(PriorityOrder.from_csv_row("10101,S18_2325"))     # <<< returns PriorityOrder
print(Order.empty(10102))
print(Order.is_valid_code("S18_1749"), Order.is_valid_code("ab"))

| | Receives | Subclass-aware | Typical use |
|---|---|---|---|
| instance method | `self` | n/a | behaviour on data |
| `@classmethod` | `cls` | **yes** | alternative constructors, registries, class-level config |
| `@staticmethod` | nothing | no | pure helper that belongs in this namespace |
| module function | nothing | no | **default choice** — don't reach for staticmethod reflexively |


---

# 6. ABCs, Protocols, `__init_subclass__`, metaclasses

## 6.1 ABC (nominal) vs Protocol (structural)

| | `abc.ABC` | `typing.Protocol` |
|---|---|---|
| Relationship | **nominal** — must inherit | **structural** — must match shape |
| Enforced | at instantiation, runtime | by the type checker |
| Third-party classes | need `register()` | work automatically |
| Shared implementation | **yes** — can hold concrete methods | no (default methods possible but unusual) |
| Coupling | tighter | looser |

**Choose Protocol when you're defining what you *consume*. Choose ABC when you're defining a base others *extend* and you want to share code.**

In [ ]:
from abc import ABC, abstractmethod

class Repository(ABC):
    """ABC: shares real implementation, forces subclasses to fill in the gaps."""

    @abstractmethod
    def get(self, id_: int) -> dict | None: ...

    @abstractmethod
    def save(self, obj: dict) -> None: ...

    # Concrete method built on the abstract ones — the value ABCs add
    def get_or_raise(self, id_: int) -> dict:
        obj = self.get(id_)
        if obj is None:
            raise KeyError(f"{type(self).__name__}: no record {id_}")
        return obj

    def upsert_many(self, objs: list[dict]) -> int:
        for o in objs:
            self.save(o)
        return len(objs)


class InMemoryRepo(Repository):
    def __init__(self): self._store: dict[int, dict] = {}
    def get(self, id_): return self._store.get(id_)
    def save(self, obj): self._store[obj["id"]] = obj

r = InMemoryRepo()
print(r.upsert_many([{"id": 1, "n": "a"}, {"id": 2, "n": "b"}]))
print(r.get_or_raise(1))
try: r.get_or_raise(99)
except KeyError as e: print("KeyError:", e)

# Instantiating an incomplete subclass fails immediately
class Broken(Repository):
    def get(self, id_): return None
try:
    Broken()
except TypeError as e:
    print("TypeError:", e)

> **ABCs fail at instantiation, not at definition.** `class Broken(Repository)` is legal; `Broken()` is not. That's deliberate — it lets you build partial hierarchies.

## 6.2 `__init_subclass__` — the modern replacement for most metaclasses

Fires when a **subclass** is created. Use it for registration, validation, and automatic configuration.

In [ ]:
class Handler:
    registry: dict[str, type["Handler"]] = {}

    def __init_subclass__(cls, /, key: str | None = None, abstract: bool = False, **kwargs):
        super().__init_subclass__(**kwargs)
        if abstract:
            return
        if key is None:
            raise TypeError(f"{cls.__name__} must declare key=...")
        if key in Handler.registry:
            raise TypeError(f"duplicate handler key {key!r}")
        if not callable(getattr(cls, "handle", None)):
            raise TypeError(f"{cls.__name__} must define handle()")
        Handler.registry[key] = cls
        cls.key = key


class CsvHandler(Handler, key="csv"):
    def handle(self, data): return f"csv: {data}"

class JsonHandler(Handler, key="json"):
    def handle(self, data): return f"json: {data}"

class AbstractBase(Handler, abstract=True):
    pass

print("registry:", Handler.registry)
print(Handler.registry["csv"]().handle("a,b,c"))

# Validation happens AT CLASS DEFINITION TIME — errors surface at import, not in prod
try:
    class NoKey(Handler): pass
except TypeError as e:
    print("TypeError:", e)

try:
    class NoHandle(Handler, key="xml"): pass
except TypeError as e:
    print("TypeError:", e)

> **Class-body keyword arguments** (`class CsvHandler(Handler, key="csv")`) are passed straight to `__init_subclass__`. This is a clean plugin-registration pattern with no metaclass and no decorator.
>
> `__init_subclass__` is **implicitly a classmethod** — don't add the decorator.

## 6.3 Metaclasses — when you actually need one

`type` is the metaclass of every class. A metaclass customises **class creation itself**.

**You need a metaclass only when you must control:** the class's own `__call__` (instantiation), the namespace *before* the class body executes (`__prepare__`), `isinstance`/`issubclass` behaviour, or the class object's own attribute access.

For everything else — registration, validation, defaults — use `__init_subclass__` or a class decorator.

In [ ]:
class SingletonMeta(type):
    """Metaclass required here because we override CLASS.__call__ — instantiation itself."""
    _instances: dict[type, object] = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class ConnectionPool(metaclass=SingletonMeta):
    def __init__(self, dsn="mysql://localhost/classicmodels"):
        print(f"  (constructing pool for {dsn})")
        self.dsn = dsn

a = ConnectionPool()
b = ConnectionPool("ignored — already built")
print("same instance:", a is b, "|", a.dsn)

In [ ]:
# A class DECORATOR does most of what people reach for metaclasses for,
# with far less machinery. Prefer this.
def add_slots_repr(cls):
    def __repr__(self):
        names = getattr(cls, "__slots__", None) or vars(self)
        return f"{cls.__name__}(" + ", ".join(f"{n}={getattr(self, n)!r}" for n in names) + ")"
    cls.__repr__ = __repr__
    return cls

@add_slots_repr
class Node:
    __slots__ = ("id", "parent")
    def __init__(self, id_, parent=None): self.id, self.parent = id_, parent

print(Node(1), Node(2, parent=1))

### Escalation ladder — reach for the lowest rung that works

```
1. plain function / module           <- start here
2. composition
3. inheritance / mixin
4. property or descriptor            <- attribute-level control
5. class decorator                   <- modify a class after creation
6. __init_subclass__ / __set_name__  <- hook into class creation
7. metaclass                         <- control instantiation or the namespace
```

If you write a metaclass, expect to justify it in code review.

---

# 7. Iterators, generators, lazy pipelines

## 7.1 The iteration protocol

`iter(x)` returns an **iterator**; `next(it)` advances it; `StopIteration` ends it. An **iterable** can be iterated many times; an **iterator** is consumed once.

In [ ]:
nums = [1, 2, 3]
it = iter(nums)
print(type(it))
print(next(it), next(it), next(it))
try: next(it)
except StopIteration: print("StopIteration")

# An iterator is its own iterator; a list is not an iterator
print(iter(it) is it)          # True
print(iter(nums) is nums)      # False

# Consequence: iterators cannot be re-used
it2 = iter(nums)
print(list(it2), list(it2))    # second call is empty

In [ ]:
class Countdown:
    """Iterable: __iter__ returns a FRESH iterator each time, so it's re-iterable."""
    def __init__(self, start): self.start = start
    def __iter__(self):
        n = self.start
        while n > 0:
            yield n            # a generator function IS an iterator factory
            n -= 1

c = Countdown(3)
print(list(c), list(c))        # re-iterable

# The two-argument form of iter(): call until a sentinel is returned
import itertools
counter = itertools.count(1).__next__
print(list(iter(counter, 5)))  # 1,2,3,4 then stops at 5

## 7.2 Generators: `yield`, `yield from`, and two-way communication

A generator function returns a generator. Calling it runs **no code** until the first `next()`.

In [ ]:
def gen():
    print("  start")
    yield 1
    print("  after first yield")
    yield 2
    print("  finishing")
    return "return value"     # becomes StopIteration.value

g = gen()
print("nothing has run yet")
print("got", next(g))
print("got", next(g))
try:
    next(g)
except StopIteration as e:
    print("StopIteration.value =", e.value)

In [ ]:
def flatten(nested):
    """yield from delegates to a sub-iterable AND propagates its return value."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)     # recursion made readable
        else:
            yield item

print(list(flatten([1, [2, [3, [4, [5]]]], 6])))

In [ ]:
def accumulator():
    """send() makes a generator a two-way channel — the basis of old-style coroutines."""
    total = 0
    while True:
        value = yield total          # yield OUT, receive IN
        if value is None:
            break
        total += value
    return total

acc = accumulator()
print(next(acc))            # prime it: runs to the first yield
print(acc.send(10))
print(acc.send(5))
print(acc.send(100))
try:
    acc.send(None)
except StopIteration as e:
    print("final total:", e.value)

> `send()`/`throw()`/`close()` are how `asyncio` drove coroutines before `async def` existed. You rarely write them now, but understanding them explains what `await` compiles to.

## 7.3 Laziness is the point: memory and short-circuiting

In [ ]:
import sys, time

N = 2_000_000

t0 = time.perf_counter()
eager = [i * i for i in range(N)]
t_eager = time.perf_counter() - t0
size_eager = sys.getsizeof(eager)

t0 = time.perf_counter()
lazy = (i * i for i in range(N))
t_lazy = time.perf_counter() - t0
size_lazy = sys.getsizeof(lazy)

print(f"list comp : {t_eager:.3f}s   {size_eager/1e6:>8.2f} MB")
print(f"generator : {t_lazy:.8f}s   {size_lazy/1e6:>8.6f} MB")

# Short-circuiting: any() stops at the first hit, so the generator never
# materialises the rest.
t0 = time.perf_counter()
found = any(i * i > 100 for i in range(N))
print(f"any() over a generator: {found} in {time.perf_counter()-t0:.8f}s")

## 7.4 A lazy ETL pipeline

Each stage is a generator. Nothing is held in memory; the whole chain processes one record at a time. This is the pattern for files larger than RAM.

In [ ]:
from collections.abc import Iterator, Iterable
import io, csv

RAW = """orderNumber,productCode,quantityOrdered,priceEach
10100,S18_1749,30,136.00
10100,S18_2248,50,55.09
10100,BAD_ROW,,
10101,S18_2325,25,108.06
10101,S18_2795,26,167.06
10102,S18_1367,39,86.13
"""

def read_rows(text: str) -> Iterator[dict]:
    yield from csv.DictReader(io.StringIO(text))

def drop_invalid(rows: Iterable[dict]) -> Iterator[dict]:
    for r in rows:
        if r["quantityOrdered"] and r["priceEach"]:
            yield r

def coerce(rows: Iterable[dict]) -> Iterator[dict]:
    for r in rows:
        yield r | {                                    # dict merge operator, 3.9+
            "orderNumber": int(r["orderNumber"]),
            "quantityOrdered": int(r["quantityOrdered"]),
            "priceEach": float(r["priceEach"]),
        }

def add_total(rows: Iterable[dict]) -> Iterator[dict]:
    for r in rows:
        yield r | {"lineTotal": r["quantityOrdered"] * r["priceEach"]}

# Compose. No intermediate lists exist.
pipeline = add_total(coerce(drop_invalid(read_rows(RAW))))

from collections import defaultdict
totals = defaultdict(float)
for row in pipeline:
    totals[row["orderNumber"]] += row["lineTotal"]

for order, total in sorted(totals.items()):
    print(f"{order}  {total:>10.2f}")

## 7.5 `itertools` — the parts worth memorising

In [ ]:
import itertools as it
from itertools import groupby, islice, chain, tee, pairwise, batched
import sys

data = [3, 1, 4, 1, 5, 9, 2, 6]

print("accumulate :", list(it.accumulate(data)))                 # running total
print("accumulate*:", list(it.accumulate(data, max)))            # running max
print("chain      :", list(chain([1, 2], [3], [4, 5])))
print("islice     :", list(islice(it.count(10), 0, 5)))          # slice any iterable
print("pairwise   :", list(pairwise(data)))                      # 3.10+ sliding pairs
print("compress   :", list(it.compress("ABCDEF", [1,0,1,0,1,0])))
print("cycle+isl  :", list(islice(it.cycle("AB"), 5)))
print("product    :", list(it.product([1,2], "ab")))
print("combos     :", list(it.combinations([1,2,3], 2)))
print("perms      :", list(it.permutations([1,2,3], 2))[:3], "...")
print("zip_longest:", list(it.zip_longest([1,2,3], "ab", fillvalue="-")))

if sys.version_info >= (3, 12):
    print("batched    :", list(batched(data, 3)))                # 3.12+ chunking
else:
    print("batched    : requires 3.12+")

In [ ]:
from itertools import groupby

# groupby REQUIRES the input to be sorted by the same key — it groups
# CONSECUTIVE runs, it is not SQL's GROUP BY.
products = [
    {"line": "Motorcycles", "name": "Harley"},
    {"line": "Classic Cars", "name": "Ford"},
    {"line": "Motorcycles", "name": "Guzzi"},
    {"line": "Classic Cars", "name": "Alfa"},
]

key = lambda p: p["line"]

print("UNSORTED -> broken:")
for k, grp in groupby(products, key=key):
    print("  ", k, [p["name"] for p in grp])

print("SORTED first -> correct:")
for k, grp in groupby(sorted(products, key=key), key=key):
    print("  ", k, [p["name"] for p in grp])

In [ ]:
from itertools import tee

# tee duplicates an iterator — but it BUFFERS, so it is not free.
# If you need two full passes, a list is usually cheaper.
src = (x * x for x in range(5))
a, b = tee(src, 2)
print(list(a), list(b))

---

# 8. Context managers

## 8.1 The protocol, and what `__exit__` can do

In [ ]:
import time

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self                      # bound to `as`
    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.t0
        print(f"[{self.label}] {self.elapsed:.4f}s")
        return False                     # False/None -> propagate any exception

with Timer("sum") as t:
    total = sum(range(1_000_000))
print("elapsed available after the block:", round(t.elapsed, 4))

In [ ]:
class Suppress:
    """Returning TRUE from __exit__ SWALLOWS the exception. Use with care."""
    def __init__(self, *exc_types): self.exc_types = exc_types
    def __enter__(self): return self
    def __exit__(self, exc_type, exc, tb):
        return exc_type is not None and issubclass(exc_type, self.exc_types)

with Suppress(ZeroDivisionError):
    1 / 0
    print("never reached")
print("execution continues")

# The stdlib version
from contextlib import suppress
with suppress(FileNotFoundError):
    open("no_such_file")
print("suppressed cleanly")

## 8.2 `@contextmanager` — the concise form

In [ ]:
from contextlib import contextmanager
import os, tempfile, pathlib

@contextmanager
def working_directory(path):
    previous = os.getcwd()
    os.chdir(path)
    try:
        yield pathlib.Path(path)        # everything before yield = __enter__
    finally:
        os.chdir(previous)              # everything in finally = __exit__

tmp = tempfile.mkdtemp()
print("before:", os.getcwd())
with working_directory(tmp) as p:
    print("inside:", os.getcwd() == os.path.realpath(str(p)) or os.getcwd())
print("after :", os.getcwd())

In [ ]:
from contextlib import contextmanager

@contextmanager
def transaction(conn):
    """The canonical DB pattern. try/except/finally maps onto __exit__."""
    conn.begin()
    try:
        yield conn
    except Exception:
        conn.rollback()
        raise                            # re-raise: never swallow silently
    else:
        conn.commit()

class FakeConn:
    def begin(self):    print("  BEGIN")
    def commit(self):   print("  COMMIT")
    def rollback(self): print("  ROLLBACK")
    def execute(self, sql): print("  ", sql)

print("success path:")
with transaction(FakeConn()) as c:
    c.execute("UPDATE customers SET creditLimit = 0")

print("failure path:")
try:
    with transaction(FakeConn()) as c:
        c.execute("UPDATE customers SET creditLimit = 0")
        raise ValueError("business rule violated")
except ValueError as e:
    print("  caught:", e)

> **The `try/except/else/finally` inside a `@contextmanager` must be complete.** If the body raises and you don't wrap the `yield`, your cleanup never runs — the generator is simply abandoned.

## 8.3 `ExitStack` — dynamic and variable numbers of resources

In [ ]:
from contextlib import ExitStack
import tempfile, os

paths = []
for i in range(3):
    fd, p = tempfile.mkstemp(suffix=f"_{i}.txt")
    os.write(fd, f"file {i}\n".encode()); os.close(fd)
    paths.append(p)

# You cannot write `with open(a), open(b), ...` when the count is dynamic.
with ExitStack() as stack:
    files = [stack.enter_context(open(p)) for p in paths]
    print([f.readline().strip() for f in files])
    # all three closed here, in reverse order, even if one raises
print("all closed:", all(f.closed for f in files))

for p in paths: os.unlink(p)

In [ ]:
from contextlib import ExitStack

# ExitStack also gives you transactional rollback of partial setup
def setup_with_rollback(fail_at: int):
    with ExitStack() as stack:
        for i in range(3):
            print(f"  acquire {i}")
            stack.callback(print, f"  release {i}")
            if i == fail_at:
                raise RuntimeError(f"failed at {i}")
        stack.pop_all()                  # success -> DON'T run the callbacks
        print("  committed")

print("failure at 1 -> partial setup unwinds:")
try: setup_with_rollback(1)
except RuntimeError as e: print("  caught:", e)

print("no failure -> pop_all keeps everything:")
setup_with_rollback(99)

## 8.4 Reentrant and reusable — the distinction that bites

In [ ]:
from contextlib import contextmanager
import threading

# A @contextmanager generator is SINGLE-USE: it cannot be re-entered.
@contextmanager
def once():
    yield "value"

cm = once()
with cm: pass
try:
    with cm: pass
except Exception as e:
    print(f"re-entering raised {type(e).__name__}: {e}")
    print("  (the generator was consumed; contextlib also deletes its stored args)")

# A class-based manager can be made reusable and reentrant. threading.RLock
# is the classic reentrant example.
lock = threading.RLock()
with lock:
    with lock:                        # same thread can re-acquire
        print("RLock re-entered fine")

plain = threading.Lock()
print("plain Lock is NOT reentrant - nesting it in one thread deadlocks")

# contextlib.contextmanager objects created by a FACTORY are fine, because
# each call builds a fresh generator:
with once(): pass
with once(): pass
print("calling the factory again works - reuse the factory, not the object")

---

# 9. Decorators and `functools`

## 9.1 `functools.wraps` is mandatory

In [ ]:
import functools, inspect

def bad(fn):
    def inner(*a, **kw): return fn(*a, **kw)
    return inner

def good(fn):
    @functools.wraps(fn)                    # copies __name__, __doc__, __wrapped__, annotations
    def inner(*a, **kw): return fn(*a, **kw)
    return inner

@bad
def f1(x: int) -> int:
    """docstring for f1"""
    return x

@good
def f2(x: int) -> int:
    """docstring for f2"""
    return x

for f in (f1, f2):
    print(f"{f.__name__:<8} doc={f.__doc__!r:<22} sig={inspect.signature(f)}")

> Without `wraps`, you break `help()`, Sphinx, `inspect.signature`, pytest fixtures, FastAPI route introspection and every framework that reads function metadata. It is not optional.

## 9.2 Parameterised decorators, and one that works with or without arguments

In [ ]:
import functools, time

def retry(_fn=None, *, attempts=3, delay=0.01, exceptions=(Exception,)):
    """Usable as @retry OR @retry(attempts=5) — the dual-form pattern."""
    def decorate(fn):
        @functools.wraps(fn)
        def inner(*a, **kw):
            last = None
            for i in range(1, attempts + 1):
                try:
                    return fn(*a, **kw)
                except exceptions as e:
                    last = e
                    print(f"  attempt {i}/{attempts} failed: {e}")
                    if i < attempts:
                        time.sleep(delay * (2 ** (i - 1)))     # exponential backoff
            raise last
        return inner
    return decorate if _fn is None else decorate(_fn)


state = {"n": 0}

@retry(attempts=4)
def flaky():
    state["n"] += 1
    if state["n"] < 3:
        raise ConnectionError(f"transient #{state['n']}")
    return "succeeded"

print(flaky())

@retry
def always_fails():
    raise ValueError("permanent")

try: always_fails()
except ValueError as e: print("gave up:", e)

## 9.3 Class-based decorators, and decorating methods

In [ ]:
import functools

class CountCalls:
    """A class-based decorator keeps state cleanly and exposes it."""
    def __init__(self, fn):
        functools.update_wrapper(self, fn)     # the wraps equivalent for classes
        self.fn = fn
        self.calls = 0

    def __call__(self, *a, **kw):
        self.calls += 1
        return self.fn(*a, **kw)

    def __get__(self, obj, objtype=None):
        # REQUIRED to work on methods: without this, `self` is never bound,
        # because the descriptor protocol is what binds methods.
        if obj is None:
            return self
        return functools.partial(self.__call__, obj)

@CountCalls
def work(x): return x * 2

work(1); work(2); work(3)
print(f"{work.__name__} called {work.calls} times")

class Service:
    @CountCalls
    def handle(self, msg): return f"handled {msg}"

s = Service()
print(s.handle("a"), s.handle("b"))
print("method calls:", Service.handle.calls)

## 9.4 `functools.cache` and `lru_cache`

In [ ]:
from functools import cache, lru_cache
import time

@cache                                # 3.9+; equivalent to lru_cache(maxsize=None), faster
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)

t0 = time.perf_counter()
print("fib(200) =", fib(200))
print(f"{time.perf_counter()-t0:.6f}s")
print(fib.cache_info())

@lru_cache(maxsize=128)               # bounded — use when the key space is large
def slow_lookup(key: str) -> str:
    time.sleep(0.05)
    return key.upper()

for k in ["a", "b", "a", "a", "c"]:
    slow_lookup(k)
print(slow_lookup.cache_info())
slow_lookup.cache_clear()

> **Three caching hazards, all of which cause production incidents:**
> 1. **Unbounded memory.** `@cache` never evicts. On a method, it also keeps `self` alive forever — a permanent leak. Use `maxsize` on anything user-driven.
> 2. **Arguments must be hashable.** No lists, dicts or sets. Convert to tuples/frozensets at the boundary.
> 3. **Stale results.** Caching an impure function (DB read, clock, RNG) caches the *first* answer for the process lifetime.

## 9.5 `singledispatch` — type-based dispatch without isinstance chains

In [ ]:
from functools import singledispatch, singledispatchmethod
from datetime import date, datetime
from decimal import Decimal

@singledispatch
def to_sql(value) -> str:
    raise TypeError(f"no SQL literal for {type(value).__name__}")

@to_sql.register
def _(value: str) -> str:  return "'" + value.replace("'", "''") + "'"

@to_sql.register
def _(value: bool) -> str: return "TRUE" if value else "FALSE"   # before int!

@to_sql.register
def _(value: int) -> str:  return str(value)

@to_sql.register(float)
@to_sql.register(Decimal)
def _(value) -> str:       return f"{value:.4f}"

@to_sql.register
def _(value: None) -> str: return "NULL"

@to_sql.register
def _(value: date) -> str: return f"'{value.isoformat()}'"

@to_sql.register
def _(value: list) -> str: return "(" + ", ".join(to_sql(v) for v in value) + ")"

for v in ["O'Brien", 42, True, 3.14159, Decimal("2.5"), None, date(2005, 5, 31),
          [1, "a", None]]:
    print(f"{type(v).__name__:<9} -> {to_sql(v)}")

In [ ]:
from functools import singledispatchmethod

class Serializer:
    @singledispatchmethod
    def dump(self, value): raise TypeError(type(value))

    @dump.register
    def _(self, value: dict): return {k: self.dump(v) for k, v in value.items()}

    @dump.register
    def _(self, value: list): return [self.dump(v) for v in value]

    @dump.register
    def _(self, value: str):  return value.strip()

    @dump.register
    def _(self, value: int):  return value

print(Serializer().dump({"a": " x ", "b": [1, " y "], "c": 3}))

> `bool` is a subclass of `int`, so its `register` must come **first** or `True` dispatches to the int branch. `singledispatch` uses MRO, and registration order matters for overlapping types.

## 9.6 Decorator order

Decorators apply **bottom-up**. `@a @b def f` means `f = a(b(f))`.

In [ ]:
def tag(name):
    def deco(fn):
        import functools
        @functools.wraps(fn)
        def inner(*a, **kw):
            print(f"  enter {name}")
            r = fn(*a, **kw)
            print(f"  exit  {name}")
            return r
        return inner
    return deco

@tag("outer")
@tag("inner")
def work(): print("  body")

work()
print("\n@cache must be OUTERMOST (above @retry), or you cache the retry wrapper\n"
      "and every retry re-executes. Order is a correctness concern, not style.")

---

# 10. Closures and scoping

## 10.1 LEGB, `nonlocal`, and cells

In [ ]:
def outer():
    count = 0
    def inner():
        nonlocal count            # rebind the ENCLOSING variable (not global)
        count += 1
        return count
    return inner

c = outer()
print(c(), c(), c())

# Closures are real objects you can inspect
print("free variables:", c.__code__.co_freevars)
print("cell contents :", [cell.cell_contents for cell in c.__closure__])

## 10.2 The late-binding trap

**This is asked in almost every senior Python interview.** Closures capture the *variable*, not its value at definition time.

In [ ]:
# THE BUG
funcs = [lambda: i for i in range(3)]
print("broken:", [f() for f in funcs])          # [2, 2, 2] — all see the final i

# FIX 1: default argument evaluates at DEFINITION time
funcs = [lambda i=i: i for i in range(3)]
print("default arg:", [f() for f in funcs])

# FIX 2: a factory creates a new scope per iteration
def make(i): return lambda: i
funcs = [make(i) for i in range(3)]
print("factory   :", [f() for f in funcs])

# FIX 3: functools.partial binds the value
from functools import partial
funcs = [partial(lambda i: i, i) for i in range(3)]
print("partial   :", [f() for f in funcs])

> The same bug appears with `for` loops registering callbacks, building `argparse` handlers, wiring GUI buttons, and constructing Spark UDFs in a loop. If a loop creates functions, check the binding.

## 10.3 Comprehension scope

In [ ]:
x = "outer"
# Comprehensions have their OWN scope (Python 3+), so this does not leak
result = [x for x in range(3)]
print(result, "| x unchanged:", x)

# But the ITERABLE expression is evaluated in the enclosing scope
class Demo:
    values = [1, 2, 3]
    doubled = [v * 2 for v in values]            # works: `values` is the iterable
    try:
        broken = [v * len(values) for v in range(3)]   # fails: class scope not visible inside
    except NameError as e:
        broken = f"NameError: {e}"

print(Demo.doubled)
print(Demo.broken)

> **Class bodies are not enclosing scopes for comprehensions.** Only the outermost iterable is evaluated in the class scope. This trips people up in class-level configuration.

## 10.4 The walrus operator — assignment expressions

In [ ]:
import re

# Avoids computing twice or restructuring the loop
data = ["a=1", "bad", "c=3"]
pattern = re.compile(r"(\w+)=(\d+)")

for item in data:
    if (m := pattern.fullmatch(item)):
        print(f"  {m.group(1)} -> {m.group(2)}")

# In a comprehension: compute once, filter and use
values = [-2, 5, -7, 3]
print([y for v in values if (y := abs(v)) > 3])

# Read-loop idiom
import io
buf = io.StringIO("l1\nl2\nl3\n")
while (line := buf.readline()):
    print("  read:", line.strip())

---

# 11. Memory model, GC, weakrefs

## 11.1 Reference counting and the cycle collector

CPython frees an object the instant its refcount hits zero. A **generational cyclic GC** exists only to break reference cycles that refcounting cannot.

In [ ]:
import sys, gc

class Node:
    def __init__(self, name): self.name = name
    def __del__(self): print(f"  freed {self.name}")

a = Node("a")
print("refcount:", sys.getrefcount(a) - 1)     # -1 for getrefcount's own argument
b = a
print("refcount:", sys.getrefcount(a) - 1)
del b
print("refcount:", sys.getrefcount(a) - 1)
del a                                          # freed immediately — no GC needed
print("---")

In [ ]:
import gc

class Node:
    def __init__(self, name): self.name = name; self.peer = None
    def __del__(self): print(f"  freed {self.name}")

# A CYCLE: refcount never reaches zero
x, y = Node("x"), Node("y")
x.peer = y
y.peer = x
del x, y
print("after del — nothing freed yet (cycle)")

collected = gc.collect()                       # the cyclic collector breaks it
print("gc.collect() reclaimed:", collected, "objects")

> **The practical rule:** cycles are collected, but *later* and *unpredictably*. If an object holds a scarce resource (file handle, socket, DB connection), rely on `with` / explicit `close()`, never on `__del__`.
>
> `__del__` on an object in a cycle also delays collection and can raise during interpreter shutdown. Avoid `__del__` in production code; use `weakref.finalize` if you truly need a hook.

## 11.2 `weakref` — caches and back-references without leaks

In [ ]:
import weakref, gc

class Session:
    def __init__(self, id_): self.id = id_
    def __repr__(self): return f"Session({self.id})"

s = Session(1)
ref = weakref.ref(s)
print("alive:", ref())
del s
gc.collect()
print("after del:", ref())            # None — the weakref did not keep it alive

In [ ]:
import weakref, gc

class Session:
    def __init__(self, id_): self.id = id_
    def __repr__(self): return f"Session({self.id})"

# A STRONG-valued cache leaks: entries live forever
strong: dict[int, Session] = {}
weak = weakref.WeakValueDictionary()

for i in range(3):
    s = Session(i)
    strong[i] = s
    weak[i] = s
    del s                              # drop the local reference

gc.collect()
print("strong cache:", dict(strong))   # all 3 — leaked
print("weak cache  :", dict(weak))     # empty — entries vanished with their objects

In [ ]:
import weakref

# The classic use: a child pointing back at its parent WITHOUT creating a cycle
class Parent:
    def __init__(self): self.children = []
    def add(self, child):
        self.children.append(child)
        child._parent = weakref.ref(self)      # weak back-reference

class Child:
    _parent = None
    @property
    def parent(self):
        return self._parent() if self._parent else None

p = Parent(); c = Child(); p.add(c)
print("child sees parent:", c.parent is p)

# weakref.finalize — a safe replacement for __del__
def cleanup(name): print(f"  finalising {name}")
obj = Parent()
weakref.finalize(obj, cleanup, "parent-obj")
del obj

## 11.3 Interning, identity and `sys.intern`

In [ ]:
import sys

# Small ints (-5..256) and compile-time-identical strings are cached.
print(256 is 256, 257 is 257)                     # implementation detail — never rely on it

a = "hello world"
b = "hello world"
print("literals   :", a is b)                     # often True (same code object)

c = "".join(["hello", " ", "world"])
print("computed   :", a is c)                     # False — built at runtime
print("interned   :", a is sys.intern(c))         # True

# Interning is a real optimisation for millions of repeated short strings
# (column names, enum-like codes) because comparison becomes pointer equality.

## 11.4 Measuring memory properly

`sys.getsizeof` is **shallow** — it reports the container, not its contents.

In [ ]:
import sys
from itertools import chain

def deep_size(obj, seen=None):
    """Recursive size. Good enough for diagnostics; use tracemalloc for real work."""
    seen = seen if seen is not None else set()
    if id(obj) in seen:
        return 0
    seen.add(id(obj))
    size = sys.getsizeof(obj)
    if isinstance(obj, dict):
        size += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_size(i, seen) for i in obj)
    elif hasattr(obj, "__dict__"):
        size += deep_size(vars(obj), seen)
    return size

nested = {"a": list(range(100)), "b": {"c": ["x" * 50] * 10}}
print(f"shallow: {sys.getsizeof(nested):>7,} bytes")
print(f"deep   : {deep_size(nested):>7,} bytes")

In [ ]:
import tracemalloc

tracemalloc.start()
snap1 = tracemalloc.take_snapshot()

# Allocate something
big = [{"id": i, "name": f"row-{i}"} for i in range(50_000)]

snap2 = tracemalloc.take_snapshot()
top = snap2.compare_to(snap1, "lineno")[:3]
for stat in top:
    print(stat)
print(f"\ncurrent / peak: {tracemalloc.get_traced_memory()[0]/1e6:.1f} / "
      f"{tracemalloc.get_traced_memory()[1]/1e6:.1f} MB")
tracemalloc.stop()
del big

**Memory reduction toolkit, in order of impact**

| Technique | Typical saving |
|---|---|
| `__slots__` / `@dataclass(slots=True)` | 30-50% per object |
| Generators instead of lists | O(n) → O(1) |
| `array.array` / NumPy for homogeneous numerics | 5-10× vs `list[int]` |
| `sys.intern` on high-repetition strings | large for millions of dupes |
| `WeakValueDictionary` for caches | removes the leak entirely |
| Category/dictionary encoding (pandas `category`) | very large on low-cardinality text |


---

# 12. Concurrency

The most consequential section for senior interviews, and where most Python engineers are weakest.

## 12.1 The GIL — what it actually is

The **Global Interpreter Lock** is a single mutex protecting CPython's internal state (chiefly reference counts). Only one thread executes Python bytecode at a time.

**What it does NOT do:**
- It does not make your code thread-safe. `counter += 1` is still a race (12.4).
- It does not block I/O parallelism. The GIL is **released** during blocking I/O.
- It does not affect C extensions that release it — NumPy, pandas, `zlib`, `hashlib`, and most DB drivers release the GIL during heavy work.

**Where the GIL is released:**

| Operation | GIL released? |
|---|---|
| socket / file read/write | **yes** |
| `time.sleep` | **yes** |
| `subprocess` wait | **yes** |
| NumPy/pandas vectorised ops | **usually yes** |
| `hashlib`, `zlib`, `bz2` on large buffers | **yes** |
| pure-Python loops, arithmetic, attribute access | **no** |

Since 3.13 the interpreter switches every 5 ms by default (`sys.getswitchinterval()`).

In [ ]:
import sys, threading
print("switch interval:", sys.getswitchinterval(), "seconds")
print("active threads :", threading.active_count())
print("GIL removable  :", hasattr(sys, "_is_gil_enabled"))

## 12.2 Choosing a model

```
Is the work waiting on something external (network, disk, DB, subprocess)?
├── YES, and there are many concurrent operations (100s-1000s)   -> asyncio
├── YES, but few operations or blocking libraries only           -> threads
└── NO — it's CPU-bound Python bytecode
    ├── the hot loop is NumPy/pandas/C                           -> threads work (GIL released)
    ├── data is large and copying is expensive                   -> shared_memory + processes
    ├── free-threaded build available and deps support it        -> threads (真 parallel)
    └── otherwise                                                -> processes
```

| | Parallel CPU | Memory | Startup | Data sharing | Scale ceiling |
|---|---|---|---|---|---|
| `threading` | no (GIL build) | shared | ~50 µs | direct, needs locks | ~100s of threads |
| `multiprocessing` | **yes** | separate | ~10-50 ms | pickle / shared_memory | ~CPU count |
| `asyncio` | no | shared | ~1 µs / task | direct, single-threaded | **10,000s of tasks** |
| free-threaded threads | **yes** | shared | ~50 µs | direct, needs locks | ~100s |
| sub-interpreters (3.14) | **yes** | mostly separate | ~ms | limited channels | ~CPU count |


### Demonstrating the GIL: CPU-bound vs I/O-bound

In [ ]:
import time, threading
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def cpu_task(n=6_000_000):
    """Pure Python arithmetic — holds the GIL."""
    total = 0
    for i in range(n):
        total += i * i
    return total

N_TASKS = 4

t0 = time.perf_counter()
[cpu_task() for _ in range(N_TASKS)]
serial = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_TASKS) as ex:
    list(ex.map(lambda _: cpu_task(), range(N_TASKS)))
threaded = time.perf_counter() - t0

print(f"CPU-bound serial  : {serial:.2f}s")
print(f"CPU-bound threads : {threaded:.2f}s   speedup {serial/threaded:.2f}x")
print("-> threads give NO speedup (often slightly worse: contention + switching)")

In [ ]:
import time, sys, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

def cpu_task(n=6_000_000):
    total = 0
    for i in range(n):
        total += i * i
    return total

# NOTEBOOK CAVEAT: worker functions must be importable by the child process.
# On Linux with the `fork` start method (Colab default through 3.13) a
# notebook-defined function works. Under `spawn` (macOS/Windows) or
# `forkserver` (Linux default from 3.14) the child re-imports __main__ and
# cannot find it -> PicklingError. Put such functions in a .py module.
try:
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4, mp_context=ctx) as ex:
        list(ex.map(cpu_task, [6_000_000] * 4))
    print(f"CPU-bound processes: {time.perf_counter()-t0:.2f}s")
    print("-> real parallelism, at the cost of process startup and pickling")
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print()
    print("Start method:", mp.get_start_method())
    print("This is the classic notebook/multiprocessing limitation.")
    print("Fix: move cpu_task into worker.py, then `from worker import cpu_task`.")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def io_task(seconds=0.4):
    """time.sleep RELEASES the GIL — stands in for a network/DB call."""
    time.sleep(seconds)
    return seconds

N = 8
t0 = time.perf_counter()
[io_task() for _ in range(N)]
serial = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as ex:
    list(ex.map(lambda _: io_task(), range(N)))
threaded = time.perf_counter() - t0

print(f"I/O-bound serial  : {serial:.2f}s")
print(f"I/O-bound threads : {threaded:.2f}s   speedup {serial/threaded:.1f}x")
print("-> near-linear speedup: the GIL is released while waiting")

## 12.3 `concurrent.futures` — the API you should default to

Prefer `ThreadPoolExecutor` / `ProcessPoolExecutor` over raw `Thread` and `Process`. You get result handling, exception propagation, and clean shutdown.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED
import time, random

def fetch(url, fail=False):
    time.sleep(random.uniform(0.05, 0.3))
    if fail:
        raise ConnectionError(f"failed: {url}")
    return f"200 {url}"

urls = [f"/api/{i}" for i in range(6)]

# submit + as_completed: results arrive in COMPLETION order
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(fetch, u, fail=(i == 2)): u for i, u in enumerate(urls)}
    for fut in as_completed(futures):
        url = futures[fut]
        try:
            print(" ", fut.result())            # re-raises inside the worker's exception
        except ConnectionError as e:
            print("  ERROR", url, "->", e)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def work(x):
    time.sleep(0.05)
    if x == 3: raise ValueError("bad input 3")
    return x * 2

# map(): results arrive in SUBMISSION order, but an exception surfaces only
# when you REACH that item — and it then TERMINATES the iterator. Two traps
# in one API.
with ThreadPoolExecutor(max_workers=4) as ex:
    it = ex.map(work, range(6))
    i = 0
    while True:
        try:
            print(f"  item {i}:", next(it))
        except StopIteration:
            print("  iterator exhausted")
            break
        except ValueError as e:
            print(f"  item {i}: ValueError: {e}")
            print("  <<< items 4 and 5 are now UNREACHABLE via this iterator")
            break
        i += 1

print()
print("Use submit() + as_completed() when you need per-item error isolation.")

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

# Timeouts and cancellation
def slow(): time.sleep(2); return "done"

with ThreadPoolExecutor(max_workers=1) as ex:
    fut = ex.submit(slow)
    try:
        fut.result(timeout=0.2)
    except TimeoutError:
        print("timed out")
    print("cancel() succeeded:", fut.cancel())   # False — already running
    print("Python cannot forcibly kill a running thread. Design for cooperative"
          " cancellation with an Event (12.5).")
    fut.result()  # wait for clean shutdown

> **Executor sizing:** `ThreadPoolExecutor` defaults to `min(32, cpu_count + 4)`. For I/O-heavy work that is often too low; for CPU-bound work threads are the wrong tool anyway. `ProcessPoolExecutor` defaults to `cpu_count()` — going higher just adds context switching.

## 12.4 Race conditions: proving `+=` is not atomic

In [ ]:
import threading

counter = 0
ITERATIONS = 200_000

def increment_unsafe():
    global counter
    for _ in range(ITERATIONS):
        counter += 1        # LOAD, ADD, STORE — the GIL can switch between them

threads = [threading.Thread(target=increment_unsafe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

expected = 4 * ITERATIONS
print(f"expected {expected:,}, got {counter:,}, lost {expected-counter:,}")
print("The GIL serialises BYTECODE, not your logical operations.")

In [ ]:
import dis
def inc():
    global counter
    counter += 1
dis.dis(inc)
print("\nFour separate bytecodes. A thread switch between LOAD and STORE loses an update.")

In [ ]:
import threading

counter = 0
lock = threading.Lock()
ITERATIONS = 200_000

def increment_safe():
    global counter
    for _ in range(ITERATIONS):
        with lock:                    # always use `with`, never manual acquire/release
            counter += 1

threads = [threading.Thread(target=increment_safe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"with lock: {counter:,} (expected {4*ITERATIONS:,})")

In [ ]:
import threading, time

# Lock granularity matters. Locking the whole loop is far faster than
# locking each iteration, because you pay acquire/release once.
ITER = 200_000

def fine_grained(store, lock):
    for _ in range(ITER):
        with lock:
            store[0] += 1

def coarse_grained(store, lock):
    local = 0
    for _ in range(ITER):
        local += 1                     # unsynchronised local work
    with lock:                         # one critical section
        store[0] += local

for name, fn in [("fine (lock per iter)", fine_grained), ("coarse (lock once)", coarse_grained)]:
    store, lock = [0], threading.Lock()
    ts = [threading.Thread(target=fn, args=(store, lock)) for _ in range(4)]
    t0 = time.perf_counter()
    for t in ts: t.start()
    for t in ts: t.join()
    print(f"{name:<24} {time.perf_counter()-t0:.3f}s  result={store[0]:,}")

## 12.5 Synchronisation primitives

| Primitive | Use for |
|---|---|
| `Lock` | mutual exclusion. **Not** reentrant |
| `RLock` | same thread may re-acquire (recursive functions, nested methods) |
| `Semaphore(n)` | cap concurrency to n (connection pools, rate limits) |
| `BoundedSemaphore(n)` | as above, but raises on over-release — catches bugs |
| `Event` | one-way signal / cooperative cancellation |
| `Condition` | wait for a predicate to become true |
| `Barrier(n)` | all n threads meet before any proceeds |
| `local()` | per-thread storage (avoids sharing entirely) |
| `queue.Queue` | **the preferred way to pass data between threads** |

In [ ]:
import threading, time, random

# Semaphore: bound concurrency without bounding the thread count
sem = threading.BoundedSemaphore(3)
active = 0
peak = 0
lock = threading.Lock()

def guarded(i):
    global active, peak
    with sem:
        with lock:
            active += 1; peak = max(peak, active)
        time.sleep(random.uniform(0.02, 0.06))
        with lock:
            active -= 1

ts = [threading.Thread(target=guarded, args=(i,)) for i in range(12)]
for t in ts: t.start()
for t in ts: t.join()
print(f"12 threads ran, peak concurrent inside the semaphore: {peak} (limit 3)")

In [ ]:
import threading, time

# Event: the correct way to cancel threads. There is no thread.kill().
stop = threading.Event()
results = []

def worker(name):
    n = 0
    while not stop.is_set():
        n += 1
        if stop.wait(timeout=0.02):        # wait() doubles as an interruptible sleep
            break
    results.append((name, n))

ts = [threading.Thread(target=worker, args=(f"w{i}",), daemon=True) for i in range(3)]
for t in ts: t.start()
time.sleep(0.15)
stop.set()                                  # cooperative shutdown signal
for t in ts: t.join(timeout=1)
print("stopped cleanly:", results)

In [ ]:
import threading

# threading.local(): each thread gets its own copy — no locking needed at all.
# This is how frameworks store per-request context.
ctx = threading.local()
seen = []

def handler(request_id):
    ctx.request_id = request_id              # invisible to other threads
    ctx.user = f"user-{request_id}"
    inner()

def inner():
    seen.append((threading.current_thread().name, ctx.request_id, ctx.user))

ts = [threading.Thread(target=handler, args=(i,), name=f"t{i}") for i in range(4)]
for t in ts: t.start()
for t in ts: t.join()
for row in sorted(seen): print(" ", row)
print("no cross-contamination — the cheapest 'lock' is not sharing")

## 12.6 Producer/consumer with `queue.Queue`

`Queue` is thread-safe and handles blocking, backpressure and shutdown. **Prefer it to shared lists plus locks.**

In [ ]:
import threading, queue, time, random

q: queue.Queue = queue.Queue(maxsize=5)      # maxsize gives you BACKPRESSURE
SENTINEL = object()
processed = []
errors = []

def producer(n_items):
    for i in range(n_items):
        q.put({"id": i, "payload": f"row-{i}"})   # blocks if the queue is full
        time.sleep(0.005)
    for _ in range(3):
        q.put(SENTINEL)                       # one sentinel per consumer

def consumer(name):
    while True:
        item = q.get()
        try:
            if item is SENTINEL:
                return
            if item["id"] == 7:
                raise ValueError("bad row 7")
            time.sleep(random.uniform(0.005, 0.02))
            processed.append((name, item["id"]))
        except Exception as e:
            errors.append((name, str(e)))
        finally:
            q.task_done()                     # required if you use q.join()

prod = threading.Thread(target=producer, args=(15,))
cons = [threading.Thread(target=consumer, args=(f"c{i}",)) for i in range(3)]
prod.start(); [c.start() for c in cons]
prod.join();  [c.join() for c in cons]

print(f"processed {len(processed)} items across {len({p[0] for p in processed})} consumers")
print("errors:", errors)

> **Sentinel pattern:** put one sentinel per consumer so each exits exactly once. The alternative — a shared `Event` plus `get(timeout=...)` — is needed when consumers must also react to shutdown while idle.
>
> **`maxsize` is not optional in production.** An unbounded queue turns a slow consumer into an out-of-memory crash. Bounding it applies backpressure to the producer instead.

## 12.7 Deadlock — cause and cure

In [ ]:
import threading, time

lock_a, lock_b = threading.Lock(), threading.Lock()
deadlocked = threading.Event()

def t1():
    with lock_a:
        time.sleep(0.05)
        acquired = lock_b.acquire(timeout=0.3)     # timeout turns a hang into an error
        if not acquired:
            deadlocked.set()
            return
        lock_b.release()

def t2():
    with lock_b:
        time.sleep(0.05)
        acquired = lock_a.acquire(timeout=0.3)
        if not acquired:
            deadlocked.set()
            return
        lock_a.release()

a, b = threading.Thread(target=t1), threading.Thread(target=t2)
a.start(); b.start(); a.join(); b.join()
print("deadlock detected via timeout:", deadlocked.is_set())

In [ ]:
import threading, time

# FIX: acquire locks in a GLOBAL CONSISTENT ORDER. Same rule as the SQL
# deadlock fix — always touch resources in the same sequence.
locks = {"a": threading.Lock(), "b": threading.Lock()}

def ordered_work(names):
    for name in sorted(names):                 # <<< the whole fix
        locks[name].acquire()
    try:
        time.sleep(0.02)
    finally:
        for name in sorted(names, reverse=True):
            locks[name].release()

ts = [threading.Thread(target=ordered_work, args=(["a", "b"],)) for _ in range(4)]
t0 = time.perf_counter()
for t in ts: t.start()
for t in ts: t.join()
print(f"no deadlock, completed in {time.perf_counter()-t0:.3f}s")

**Four deadlock defences, in order of preference**

1. **Don't share** — `threading.local`, message passing via `Queue`, immutable data
2. **One lock** — coarser but correct beats fine-grained and broken
3. **Global lock ordering** — sort by a stable key before acquiring
4. **Timeouts** — `acquire(timeout=...)`; converts a silent hang into a loud error you can retry

## 12.8 Processes: pickling, and shared memory

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time, os, multiprocessing as mp

def worker(chunk):
    """Must be importable by the child (module-level in a .py file, ideally).
    Lambdas and closures can NEVER be sent to a process pool."""
    return sum(x * x for x in chunk), os.getpid()

chunks = [range(i * 300_000, (i + 1) * 300_000) for i in range(4)]
ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None

try:
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4, mp_context=ctx) as ex:
        results = list(ex.map(worker, chunks))
    print(f"{time.perf_counter()-t0:.2f}s")
    print("distinct PIDs:", len({pid for _, pid in results}), "| parent:", os.getpid())
    print("total:", sum(s for s, _ in results))
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("See the note in 12.2: move worker() into a .py module for spawn/forkserver.")

In [ ]:
from concurrent.futures import ProcessPoolExecutor

# What cannot cross a process boundary
def show_pickle_limits():
    import pickle
    cases = {
        "module function": show_pickle_limits,
        "lambda":          lambda x: x,
        "closure":         (lambda: (lambda y: y))(),
        "open file":       None,
        "generator":       (i for i in range(3)),
    }
    for name, obj in cases.items():
        if obj is None:
            print(f"  {name:<16} -> not picklable (file handles, sockets, locks)")
            continue
        try:
            pickle.dumps(obj)
            print(f"  {name:<16} -> picklable")
        except Exception as e:
            print(f"  {name:<16} -> {type(e).__name__}")

show_pickle_limits()
print("\nWorkarounds: module-level functions, functools.partial,")
print("or pass configuration rather than live objects.")

In [ ]:
from multiprocessing import shared_memory
import numpy as np

# shared_memory avoids pickling large arrays entirely — the workers map the
# SAME physical pages. This is how you parallelise over GB-scale data.
arr = np.arange(1_000_000, dtype=np.int64)

shm = shared_memory.SharedMemory(create=True, size=arr.nbytes)
view = np.ndarray(arr.shape, dtype=arr.dtype, buffer=shm.buf)
view[:] = arr[:]

print(f"array   : {arr.nbytes/1e6:.1f} MB")
print(f"shm name: {shm.name}  (workers attach by name, zero copy)")
print("first 5 :", view[:5])

# Workers would do:  shared_memory.SharedMemory(name=shm.name)
shm.close()
shm.unlink()          # unlink exactly ONCE, or you leak /dev/shm segments
print("cleaned up")

> **Start methods matter.** `fork` (Linux default through 3.13) is fast but unsafe with threads — a forked child can inherit a held lock and deadlock. `spawn` (macOS/Windows default) is safe but re-imports your module, so top-level side effects run again. Python 3.14 changes the Linux default to `forkserver`.
>
> **Always guard entry points** with `if __name__ == "__main__":` — mandatory under `spawn`/`forkserver`.

## 12.9 asyncio — modern structured concurrency

One thread, one event loop, cooperative multitasking. Concurrency without locks, scaling to tens of thousands of in-flight operations.

**The rule that matters: never block the event loop.** One synchronous `time.sleep` or CPU-heavy loop stalls every task.

In [ ]:
import asyncio, time, random

async def fetch(name, delay):
    await asyncio.sleep(delay)              # yields control back to the loop
    return f"{name} in {delay:.2f}s"

async def main():
    t0 = time.perf_counter()
    # TaskGroup (3.11+) is the modern replacement for gather():
    # structured, cancels siblings on failure, and always awaits children.
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(fetch(f"req{i}", random.uniform(0.1, 0.4)))
                 for i in range(6)]
    for t in tasks:
        print(" ", t.result())
    print(f"total {time.perf_counter()-t0:.2f}s (not the sum of delays)")

await main()          # in Colab/Jupyter the loop is already running, so `await` directly

> **In a notebook you `await main()` directly.** In a script you call `asyncio.run(main())`. Calling `asyncio.run` inside a running loop raises `RuntimeError`.

In [ ]:
import asyncio

async def flaky(i):
    await asyncio.sleep(0.05 * i)
    if i == 3:
        raise ValueError(f"task {i} failed")
    return i

async def with_taskgroup():
    try:
        async with asyncio.TaskGroup() as tg:
            for i in range(6):
                tg.create_task(flaky(i))
    except* ValueError as eg:                     # ExceptionGroup — see section 14
        print("  TaskGroup raised:", [str(e) for e in eg.exceptions])
    print("  all sibling tasks were cancelled and awaited — no leaks")

async def with_gather():
    results = await asyncio.gather(*[flaky(i) for i in range(6)],
                                   return_exceptions=True)
    print("  gather results:", [type(r).__name__ if isinstance(r, Exception) else r
                                for r in results])
    print("  note: siblings kept running to completion")

print("TaskGroup (fail-fast, structured):"); await with_taskgroup()
print("gather (collect everything):");        await with_gather()

| | `asyncio.gather` | `asyncio.TaskGroup` (3.11+) |
|---|---|---|
| On child failure | others keep running | **others cancelled** |
| Error surface | first exception, or a list | `ExceptionGroup` |
| Orphaned tasks | possible | impossible |
| Use when | you want every result regardless | **default choice** |


In [ ]:
import asyncio, time

# Timeouts: asyncio.timeout (3.11+) is a context manager, so it composes
async def slow(): await asyncio.sleep(1.0); return "done"

async def demo_timeout():
    try:
        async with asyncio.timeout(0.2):
            await slow()
    except TimeoutError:
        print("  timed out — the task was CANCELLED, not abandoned")

    # Bound concurrency with a Semaphore, exactly like threads
    sem = asyncio.Semaphore(3)
    peak = active = 0
    async def guarded(i):
        nonlocal peak, active
        async with sem:
            active += 1; peak = max(peak, active)
            await asyncio.sleep(0.05)
            active -= 1

    async with asyncio.TaskGroup() as tg:
        for i in range(12):
            tg.create_task(guarded(i))
    print(f"  12 tasks, peak concurrency {peak} (limit 3)")

await demo_timeout()

In [ ]:
import asyncio, time

# THE CARDINAL SIN: blocking the loop
def blocking_cpu():
    total = 0
    for i in range(4_000_000):
        total += i
    return total

async def bad():
    t0 = time.perf_counter()
    async with asyncio.TaskGroup() as tg:
        for _ in range(3):
            tg.create_task(asyncio.sleep(0.2))
        blocking_cpu()                       # <<< stalls every task
    return time.perf_counter() - t0

async def good():
    t0 = time.perf_counter()
    async with asyncio.TaskGroup() as tg:
        for _ in range(3):
            tg.create_task(asyncio.sleep(0.2))
        # to_thread offloads blocking work; the loop keeps running
        tg.create_task(asyncio.to_thread(blocking_cpu))
    return time.perf_counter() - t0

print(f"blocking in the loop : {await bad():.2f}s")
print(f"offloaded to a thread: {await good():.2f}s")

**Bridging sync and async**

| Need | Use |
|---|---|
| Run blocking I/O from async code | `await asyncio.to_thread(fn, *args)` |
| Run CPU-bound work from async code | `loop.run_in_executor(ProcessPoolExecutor(), fn, ...)` |
| Call async code from sync code | `asyncio.run(coro())` (only outside a loop) |
| Long-lived background task | `tg.create_task(...)` inside a TaskGroup; never a bare `create_task` you don't await |

> **Never fire-and-forget.** A `Task` with no reference can be garbage-collected mid-flight, and its exception disappears. Keep a reference or use a TaskGroup.

In [ ]:
import asyncio

# Async iterators and async context managers
class RateLimitedFeed:
    def __init__(self, n, delay=0.02):
        self.n, self.delay, self.i = n, delay, 0

    async def __aenter__(self):
        print("  feed opened"); return self

    async def __aexit__(self, *exc):
        print("  feed closed"); return False

    def __aiter__(self): return self

    async def __anext__(self):
        if self.i >= self.n:
            raise StopAsyncIteration
        await asyncio.sleep(self.delay)
        self.i += 1
        return f"item-{self.i}"

async def consume():
    async with RateLimitedFeed(4) as feed:
        async for item in feed:
            print("   got", item)
    # async generator equivalent
    async def agen(n):
        for i in range(n):
            await asyncio.sleep(0.01)
            yield i
    print("  async comprehension:", [x async for x in agen(4)])

await consume()

## 12.10 Free-threaded Python (3.13 experimental, 3.14 officially supported)

**PEP 703** removes the GIL. **PEP 779** made the free-threaded build officially supported in Python 3.14 (released 7 October 2025).

**Key facts for interviews:**

- It is a **separate build**, not a flag: `python3.14t`. `pip install` needs wheels built for it.
- Single-threaded overhead fell from roughly 40% (3.13) to about **5-10%** (3.14), because the specialising adaptive interpreter was re-enabled.
- CPU-bound threaded code can now scale across cores — the reason to care.
- **Your code still needs locks.** Removing the GIL removes accidental protection; it does not make `counter += 1` atomic. If anything, races that the GIL masked become more likely.
- C extensions must opt in. Any extension that hasn't will re-enable the GIL at import, silently removing the benefit.
- Some things are explicitly not thread-safe even in 3.14: sharing one iterator across threads, and reading `frame.f_locals` of a frame executing in another thread.

In [ ]:
import sys

if hasattr(sys, "_is_gil_enabled"):
    print("free-threading-capable build")
    print("GIL currently enabled:", sys._is_gil_enabled())
    print()
    print("If False, the cpu_task threading benchmark in 12.2 would show real speedup.")
else:
    print(f"Standard GIL build (Python {sys.version.split()[0]}).")
    print()
    print("To try free-threading:")
    print("  uv python install 3.14t     # or the installer's free-threaded option")
    print("  python3.14t -c 'import sys; print(sys._is_gil_enabled())'")

checklist = [
    "1. Audit every shared mutable structure; add explicit locks",
    "2. Assume NOTHING is atomic - not dict updates, not append ordering",
    "3. Verify every C extension has a free-threaded wheel",
    "4. Benchmark single-threaded performance too (you pay 5-10%)",
    "5. Prefer immutable data and message passing over locks where you can",
]
print()
print("Migration checklist for free-threaded builds")
print("-" * 44)
for line in checklist:
    print(line)

## 12.11 Sub-interpreters (Python 3.14)

Python 3.14 exposes **`concurrent.interpreters`** (PEP 734) in the standard library — multiple isolated interpreter states inside one process.

**The positioning:** a middle ground between threads (shared everything) and processes (share nothing).

| | Threads | Sub-interpreters | Processes |
|---|---|---|---|
| Memory isolation | none | **mostly isolated** | full |
| Parallel CPU | no (GIL build) | **yes** — own GIL each | yes |
| Startup cost | µs | ms | tens of ms |
| Object sharing | direct | limited channels only | pickle |
| Crash blast radius | whole process | whole process | one worker |

Honest caveat: the 3.14 API is deliberately minimal. There aren't yet high-level mechanisms for sharing objects, and it doesn't behave much like threads *or* subprocesses. Treat it as promising for plugin isolation and CPU fan-out, not yet as a drop-in replacement.

In [ ]:
import sys

if sys.version_info >= (3, 14):
    from concurrent import interpreters
    interp = interpreters.create()
    interp.exec("import math; print('  from sub-interpreter:', math.factorial(10))")
    print("interpreter id:", interp.id)
    interp.close()
else:
    print(f"concurrent.interpreters requires Python 3.14+ (running {sys.version.split()[0]}).")
    print()
    print("Shape of the 3.14 API:")
    for line in [
        "from concurrent import interpreters",
        "interp = interpreters.create()",
        "interp.exec('import mymodule; mymodule.run()')",
        "interp.close()",
    ]:
        print("   ", line)
    print()
    print("Each interpreter has its own GIL, so CPU-bound work runs in parallel")
    print("without the pickling cost of multiprocessing.")

## 12.12 Concurrency decision summary

| Workload | Choose | Why |
|---|---|---|
| 1000s of HTTP calls | **asyncio** | µs per task, one thread, no lock complexity |
| A handful of DB queries with a blocking driver | **threads** | GIL released on I/O; no rewrite needed |
| CPU-bound pure Python | **processes** | only way to use cores on a GIL build |
| CPU-bound NumPy/pandas | **threads** | those libraries release the GIL |
| Large arrays, many workers | **processes + shared_memory** | avoids pickling GB of data |
| Mixed async service with occasional CPU work | **asyncio + run_in_executor** | keeps the loop responsive |
| CPU-bound, all deps have 3.14t wheels | **free-threaded threads** | shared memory *and* parallelism |
| Untrusted plugins needing parallelism | **sub-interpreters (3.14)** | isolation without process cost |

**Rules that hold regardless of model**

1. Measure before choosing. "It's slow" is not a diagnosis of CPU vs I/O.
2. Bound everything — pool sizes, queue depths, semaphores. Unbounded concurrency is how services fall over.
3. Prefer not sharing. `threading.local`, queues, immutable data.
4. Design cancellation in from the start; you cannot kill a thread.
5. Never block the event loop.
6. Removing the GIL does not remove the need for locks.

---

# 13. Structural pattern matching (3.10+)

`match`/`case` is **destructuring**, not a switch statement. Its value is matching *shape* — nested dicts, ASTs, event payloads, tagged unions.

In [ ]:
def handle(event: dict) -> str:
    match event:
        # Mapping pattern: matches if these keys exist; extra keys are IGNORED
        case {"type": "order", "id": int(order_id), "lines": [*lines]} if len(lines) > 1:
            return f"multi-line order {order_id} with {len(lines)} lines"

        case {"type": "order", "id": int(order_id)}:
            return f"order {order_id}"

        # Literal alternatives with |
        case {"type": "payment" | "refund" as kind, "amount": float(amt) | int(amt)}:
            return f"{kind} of {amt:.2f}"

        # Capture the whole subject
        case {"type": str(unknown)}:
            return f"unhandled type {unknown!r}"

        case _:
            return "malformed"

for e in [
    {"type": "order", "id": 10100, "lines": ["a", "b", "c"]},
    {"type": "order", "id": 10101, "lines": ["a"]},
    {"type": "payment", "amount": 1200},
    {"type": "refund", "amount": 99.5},
    {"type": "shipment"},
    {"nope": 1},
]:
    print(f"  {str(e):<58} -> {handle(e)}")

In [ ]:
from dataclasses import dataclass

# Class patterns with __match_args__ (dataclasses get it free) — positional matching
@dataclass
class Point:  x: float; y: float
@dataclass
class Circle: centre: Point; radius: float
@dataclass
class Rect:   topleft: Point; w: float; h: float

def describe(shape) -> str:
    match shape:
        case Circle(centre=Point(0, 0), radius=r):
            return f"circle at origin, r={r}"
        case Circle(centre=Point(x, y), radius=r) if r > 10:
            return f"large circle at ({x}, {y}), r={r}"
        case Circle(Point(x, y), r):                       # positional
            return f"circle at ({x}, {y}), r={r}"
        case Rect(Point(x, y), w, h) if w == h:
            return f"square at ({x}, {y}), side={w}"
        case Rect(topleft=Point(x, y), w=w, h=h):
            return f"rect at ({x}, {y}), {w}x{h}"
        case _:
            return "unknown shape"

for s in [Circle(Point(0, 0), 5), Circle(Point(3, 4), 20), Circle(Point(1, 1), 2),
          Rect(Point(0, 0), 4, 4), Rect(Point(2, 2), 6, 3), "triangle"]:
    print(f"  {describe(s)}")

In [ ]:
# Sequence patterns: * captures the rest. Note str/bytes do NOT match
# sequence patterns — a deliberate design choice to avoid surprises.
def parse_command(argv: list[str]) -> str:
    match argv:
        case []:                       return "no command"
        case ["help"]:                 return "show help"
        case ["run", script]:          return f"run {script}"
        case ["run", script, *flags]:  return f"run {script} with {flags}"
        case [cmd, *_] if cmd.startswith("-"): return f"unexpected flag {cmd}"
        case [cmd, *_]:                return f"unknown command {cmd}"

for argv in [[], ["help"], ["run", "etl.py"], ["run", "etl.py", "--dry", "-v"],
             ["--verbose"], ["deploy", "prod"]]:
    print(f"  {str(argv):<34} -> {parse_command(argv)}")

print("\n  'abc' matches [str()] ?", end=" ")
match "abc":
    case [*_]: print("yes")
    case str(): print("no — str is excluded from sequence patterns")

> **Two traps.** A bare name in a pattern **captures**, it does not compare: `case STATUS:` binds everything to `STATUS`. To compare against a constant you need a dotted name (`case Status.OPEN:`) or a literal.
>
> And mapping patterns are **partial** — `case {"a": 1}` matches `{"a": 1, "b": 2}`. Use `**rest` to capture the remainder, or check `len()` in a guard, if you need exactness.

In [ ]:
from enum import Enum

class Status(Enum):
    OPEN = "open"; CLOSED = "closed"

value = Status.CLOSED

# WRONG: OPEN is a capture pattern — it matches ANYTHING and rebinds
OPEN = Status.OPEN
match value:
    case OPEN:
        print("  matched 'OPEN' — but OPEN is now", OPEN, "<<< bug")

# RIGHT: dotted names are value patterns
match value:
    case Status.OPEN:   print("  open")
    case Status.CLOSED: print("  closed  <<< correct")

---

# 14. Exception groups and `except*` (3.11+)

Before 3.11, a function could only raise one exception. `ExceptionGroup` lets concurrent operations report **all** their failures, and `except*` filters them by type.

In [ ]:
def validate_all(rows: list[dict]) -> None:
    errors = []
    for i, row in enumerate(rows):
        if "id" not in row:
            errors.append(KeyError(f"row {i}: missing id"))
        elif not isinstance(row.get("qty"), int):
            errors.append(TypeError(f"row {i}: qty must be int"))
        elif row["qty"] < 0:
            errors.append(ValueError(f"row {i}: qty negative"))
    if errors:
        raise ExceptionGroup("validation failed", errors)

rows = [
    {"id": 1, "qty": 5},
    {"qty": 3},
    {"id": 3, "qty": "ten"},
    {"id": 4, "qty": -2},
]

try:
    validate_all(rows)
except* KeyError as eg:
    print("missing fields :", [str(e) for e in eg.exceptions])
except* TypeError as eg:
    print("type errors    :", [str(e) for e in eg.exceptions])
except* ValueError as eg:
    print("value errors   :", [str(e) for e in eg.exceptions])

> **`except*` clauses are not mutually exclusive.** Unlike `except`, *every* matching clause runs — the group is split by type and each handler receives its own subgroup. Unhandled parts are re-raised as a smaller group.

In [ ]:
# add_note (3.11+): attach context to an exception without wrapping it
def load_config(path):
    try:
        raise FileNotFoundError(path)
    except FileNotFoundError as e:
        e.add_note(f"looked in: {path}")
        e.add_note("hint: set CONFIG_PATH or create the file")
        raise

try:
    load_config("/etc/app/config.toml")
except FileNotFoundError as e:
    print(type(e).__name__, e)
    for note in getattr(e, "__notes__", []):
        print("  note:", note)

In [ ]:
# Exception chaining: `raise ... from ...` preserves the cause
def parse_row(raw: str) -> int:
    try:
        return int(raw)
    except ValueError as e:
        raise ValueError(f"bad row: {raw!r}") from e     # __cause__ set

try:
    parse_row("abc")
except ValueError as e:
    print("raised :", e)
    print("cause  :", repr(e.__cause__))

# `from None` SUPPRESSES the original — use when the internal error is noise
def lookup(d, k):
    try:
        return d[k]
    except KeyError:
        raise LookupError(f"no such key {k!r}") from None

try:
    lookup({}, "x")
except LookupError as e:
    print("suppressed cause:", e.__cause__, "| context still set:", type(e.__context__).__name__)

| Form | `__cause__` | Traceback shows |
|---|---|---|
| `raise X` inside `except` | not set | "During handling of the above exception..." |
| `raise X from e` | set to `e` | "The above exception was the direct cause..." |
| `raise X from None` | `None` | only X |

Use `from e` when the original diagnoses the problem; `from None` when it's an implementation detail leaking through your API.

---

# 15. Performance engineering

**Order of operations: measure, find the hot spot, fix the algorithm, then micro-optimise — and only if it's still hot.**

## 15.1 Profiling

In [ ]:
import cProfile, pstats, io

def build_index(rows):
    index = {}
    for r in rows:
        key = r["country"]
        if key not in index:
            index[key] = []
        index[key].append(r["id"])
    return index

def slow_lookup(rows, ids):
    # O(n*m): a linear scan per lookup
    return [r for i in ids for r in rows if r["id"] == i]

rows = [{"id": i, "country": ["USA", "France", "Japan"][i % 3]} for i in range(3000)]
ids = list(range(0, 3000, 100))

pr = cProfile.Profile()
pr.enable()
build_index(rows)
slow_lookup(rows, ids)
pr.disable()

s = io.StringIO()
pstats.Stats(pr, stream=s).sort_stats("cumulative").print_stats(6)
print(s.getvalue()[:1400])

In [ ]:
import timeit

# timeit for micro-benchmarks — it handles warmup and repetition
setup = "rows = [{'id': i} for i in range(3000)]; ids = list(range(0, 3000, 100))"

linear = timeit.timeit(
    "[r for i in ids for r in rows if r['id'] == i]", setup=setup, number=20)

indexed = timeit.timeit(
    "[idx[i] for i in ids]",
    setup=setup + "; idx = {r['id']: r for r in rows}", number=20)

print(f"linear scan  : {linear:.4f}s")
print(f"dict index   : {indexed:.6f}s")
print(f"speedup      : {linear/indexed:,.0f}x   <<< ALGORITHM, not micro-optimisation")

## 15.2 Complexity of the built-in types

This table answers most "why is it slow" questions.

| Operation | list | dict / set | deque |
|---|---|---|---|
| index / key access | O(1) | O(1) | O(1) ends, O(n) middle |
| `in` (membership) | **O(n)** | **O(1)** | O(n) |
| append / add | O(1) amortised | O(1) | O(1) both ends |
| `insert(0, x)` / `pop(0)` | **O(n)** | n/a | **O(1)** |
| delete by value | O(n) | O(1) by key | O(n) |
| sort | O(n log n) | n/a | n/a |
| min / max | O(n) | O(n) | O(n) |

> **The two most common real-world fixes:** replacing `x in some_list` with a set, and replacing `list.pop(0)` with `collections.deque`.

In [ ]:
import timeit
from collections import deque

n = 20_000
print("pop(0) vs deque.popleft():")
t_list = timeit.timeit("while l: l.pop(0)", setup=f"l=list(range({n}))", number=1)
t_deq  = timeit.timeit("while d: d.popleft()",
                       setup=f"from collections import deque; d=deque(range({n}))", number=1)
print(f"  list.pop(0)     : {t_list:.4f}s   O(n) per pop")
print(f"  deque.popleft() : {t_deq:.4f}s   O(1) per pop")
print(f"  speedup         : {t_list/t_deq:,.0f}x")

## 15.3 Micro-optimisations that actually matter

Only reach for these once profiling says the loop itself is hot.

In [ ]:
import timeit

# 1. Local variable lookup beats global/attribute lookup (LOAD_FAST vs LOAD_GLOBAL)
setup = "import math"
glob = timeit.timeit("[math.sqrt(i) for i in range(2000)]", setup=setup, number=500)
locl = timeit.timeit("[sqrt(i) for i in range(2000)]",
                     setup="from math import sqrt", number=500)
print(f"math.sqrt (global+attr): {glob:.4f}s")
print(f"sqrt (local name)      : {locl:.4f}s   {glob/locl:.2f}x faster")

In [ ]:
import timeit

# 2. Build strings with join, never += in a loop
n = 20_000
plus = timeit.timeit("s=''\nfor w in words: s += w",
                     setup=f"words=['x']*{n}", number=5)
join = timeit.timeit("''.join(words)", setup=f"words=['x']*{n}", number=5)
print(f"+= in a loop : {plus:.4f}s")
print(f"''.join()    : {join:.6f}s   {plus/join:,.0f}x faster")

In [ ]:
import timeit

# 3. Comprehensions beat append loops (the append lookup is hoisted away)
loop = timeit.timeit("out=[]\nfor i in range(5000): out.append(i*i)", number=500)
comp = timeit.timeit("[i*i for i in range(5000)]", number=500)
gen  = timeit.timeit("sum(i*i for i in range(5000))", number=500)
print(f"append loop     : {loop:.4f}s")
print(f"list comp       : {comp:.4f}s   {loop/comp:.2f}x")
print(f"generator + sum : {gen:.4f}s   (no list built at all)")

In [ ]:
import timeit

# 4. Batch, don't loop, when a C-level builtin exists
py  = timeit.timeit("total=0\nfor x in data: total += x", setup="data=list(range(50000))", number=200)
blt = timeit.timeit("sum(data)", setup="data=list(range(50000))", number=200)
print(f"python loop : {py:.4f}s")
print(f"sum()       : {blt:.4f}s   {py/blt:.1f}x  (the loop runs in C)")

try:
    import numpy as np
    npy = timeit.timeit("arr.sum()", setup="import numpy as np; arr=np.arange(50000)", number=200)
    print(f"numpy .sum(): {npy:.4f}s   {py/npy:.0f}x")
except ImportError:
    pass

### The hierarchy of speedups

| Level | Typical gain | Example |
|---|---|---|
| **Algorithm / data structure** | 10-10,000× | list scan → dict lookup |
| **Avoid the work** | ∞ | caching, early exit, laziness |
| **Move the loop into C** | 10-100× | `sum()`, NumPy, pandas vectorisation |
| **Parallelise** | up to core count | processes, free-threading |
| **Micro-optimise Python** | 1.2-3× | local names, comprehensions |
| **Rewrite in Cython/Rust** | 10-100× | last resort, high maintenance cost |

Work top-down. Engineers who start at the bottom waste weeks for 20%.

---

# 16. Tooling — the modern stack

| Concern | Use | Replaces |
|---|---|---|
| Env + dependency + Python install | **uv** | pip, virtualenv, pyenv, pip-tools, poetry |
| Lint + format | **ruff** | flake8, black, isort, pyupgrade, pylint (mostly) |
| Type check | **mypy** or **pyright**/basedpyright | — |
| Test | **pytest** | unittest |
| Project metadata | **pyproject.toml** | setup.py, setup.cfg |
| Build/publish | **uv build** / hatchling | setuptools invocations |
| Task running | **just** / make / `uv run` | shell scripts |
| Pre-commit | **pre-commit** with ruff + mypy hooks | — |

A minimal `pyproject.toml` a reviewer would accept:

```toml
[project]
name = "myservice"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = ["httpx>=0.27", "pydantic>=2.7"]

[dependency-groups]
dev = ["pytest>=8", "ruff>=0.6", "mypy>=1.11"]

[tool.ruff]
line-length = 100
target-version = "py312"

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM", "RUF", "ASYNC", "PERF"]

[tool.mypy]
python_version = "3.12"
strict = true
warn_unreachable = true

[tool.pytest.ini_options]
addopts = "-q --strict-markers"
```

Ruff rule sets worth knowing: `UP` (pyupgrade — rewrites old idioms to modern ones), `B` (bugbear — real bug patterns including mutable defaults), `SIM` (simplify), `ASYNC` (blocking calls in async code), `PERF` (performance anti-patterns).

In [ ]:
# What a strict type checker would flag in this notebook's own bad examples.
# These run fine at runtime — that's the point of static checking.
BAD = '''
def f(x: int) -> str:
    return x                      # error: incompatible return type

def g(items: list = []):          # ruff B006: mutable default
    items.append(1)

def h(a: int | None):
    return a + 1                  # error: unsupported operand for None

import typing
def old(x: typing.List[int]): ... # ruff UP006: use list[int]
'''
print("mypy/ruff would flag:")
for line in ["incompatible return type (int vs str)",
             "B006 mutable default argument",
             "None not narrowed before arithmetic",
             "UP006 deprecated typing.List"]:
    print("  -", line)

---

# 17. Interview questions (SDE-3 / senior)

Answer each out loud in under 90 seconds. If you can't, revisit the section.

## Object model and OOP

1. Walk me through what happens when Python evaluates `obj.attr`.
2. Why can't an instance attribute shadow a `@property`, but it *can* shadow a method?
3. What is `super()` actually resolving against, and why does the diamond call `A.__init__` only once?
4. Write a descriptor that validates its attribute. Why does `__set_name__` exist?
5. `cached_property` — how does the caching work mechanically, and why does it fail with `__slots__`?
6. When do you need a metaclass rather than `__init_subclass__`?
7. `ABC` vs `Protocol` — which for a function parameter, which for a shared base, and why?
8. What does `__slots__` cost you? Name three things it breaks.
9. The `__eq__`/`__hash__` contract — what breaks if you violate it, and how?
10. `classmethod` vs `staticmethod` vs a module-level function. Defend your default.

## Concurrency

11. What does the GIL protect, and name four operations that release it.
12. Show me why `counter += 1` races despite the GIL. Reference the bytecode.
13. `TaskGroup` vs `gather` — behavioural difference on child failure.
14. I have 5,000 concurrent HTTP calls. Threads, processes or asyncio? Now the payloads are 200 MB NumPy arrays and the work is CPU-bound — what changes?
15. How do you cancel a running thread? (Trick question.)
16. Four ways to prevent deadlock, in order of preference.
17. Free-threaded Python removes the GIL — do you still need locks? Why?
18. What cannot be sent to a `ProcessPoolExecutor` worker, and how do you work around it?
19. Why must a production `queue.Queue` have a `maxsize`?
20. Something blocks the event loop in a running asyncio service. How do you find it, and how do you fix it?

## Language mechanics

21. Explain the late-binding closure bug and three fixes.
22. Generator vs iterator vs iterable. What does `send()` do?
23. Reference counting vs the cyclic GC — when is each responsible for freeing?
24. When would you reach for `weakref`?
25. Why is `functools.wraps` not optional?
26. Three ways `@cache` causes a production incident.
27. `except*` — why can multiple clauses fire for one raise?
28. `raise X from e` vs `raise X from None` — when each?
29. In a `match` statement, why does `case SOME_NAME:` match everything?
30. Rank these speedup techniques by expected gain and justify the order.

## Design and judgement

31. A colleague adds a metaclass to register plugins. What do you say in review?
32. Inheritance or composition here — walk me through your decision framework.
33. When do you introduce Pydantic, and where in the system does validation belong?
34. `frozen=True, slots=True` dataclass vs `NamedTuple` vs `TypedDict` — pick one for three scenarios.
35. Service memory grows steadily and never falls. Give me your diagnostic sequence.

---

## Final summary

**The five ideas that unlock everything else**

1. **Everything is protocols.** Behaviour comes from dunder methods, not from inheritance. Design by implementing protocols.
2. **Descriptors explain attribute access.** Methods, properties, `cached_property`, ORM fields and dataclass fields are all one mechanism.
3. **`super()` follows the MRO of the instance**, not the class you're writing. That is what makes cooperative multiple inheritance possible.
4. **The GIL serialises bytecode, not logic.** It gives you nothing for free, and removing it changes nothing about your need for locks.
5. **Measure, then fix the algorithm.** Micro-optimisation is the last 20%, not the first.

**And the one habit that separates senior from mid-level:** knowing *which* concurrency model the problem needs before writing any of it — because that decision is expensive to reverse.